# Notebook 11 — Logistic Regression Representation Search

Notebook 8–10 已经完成了 **Source Space Search**：

```text
NB8  Atomic Evidence
        ↓
NB9  Ranking / Direction Construction
        ↓
NB10 Wrapper Search
        ↓
LR Core Source Set + Transformation Rescue Pool
```

Notebook 11 在 NB10 所选 source space 上比较不同的 Logistic Regression 输入表示方法，重点评估候选表示对预测能力、时间稳定性和模型复杂度的影响。

这一步对 LR 很重要。Tree 可以通过 split 自动表达 threshold、nonlinearity 和 interaction；
但 LR 本身主要学习线性组合，因此输入表示方式会直接影响它能表达什么关系。

本册只保留两条清楚的路线：

```text
A. Local / interpretable representation
   raw X
   log(X)
   winsor(X)
   bin(X)
   spline(X)
   A / B
   (A-B) / |B|
   A × B

B. Global / compressed representation
   PCA on numeric Core
```

两条路线解决的问题不同：

```text
Local FE
→ 某一个 source，或某一对 sources，是否应该换一种形式进入 LR？

PCA
→ 整块 numeric Core 是否可以用更少的 latent components 表示？
```

分析流程：

```text
NB10 Core + Rescue
        ↓
1. Source Audit
        ↓
2. 生成受约束的 representation candidates
        ↓
3. Cheap Probe：先淘汰没有增量迹象的候选
        ↓
4. 组成少量 Representation Packages
        ↓
5. Raw Core vs Local FE vs PCA
        ↓
6. F1–F4 Temporal Robustness
        ↓
7. Final Representation Comparison
        ↓
8. Freeze representation → Notebook 12
```

本册仍然 **不读取**：

```text
Validation Week 64–77
OOT Week 78–91
```

Notebook 11 只负责 representation search；最终 LR 的正则化强度、penalty、solver 等模型配置继续留给 Notebook 12。


## 1. 上游特征空间与表示学习的分工

Notebook 10 已经完成了 raw source space 的 wrapper search，并给出最终推荐解。

对 LR 来说，这个推荐并不等于“最终 design matrix 已经完成”。原因是：

```text
NB10
→ 判断哪些 raw sources 值得继续保留

NB11
→ 判断这些 sources 应该以什么形式进入 LR
```

因此 NB11 接收两类 source：

```text
Core
→ raw representation 已经通过 NB10 的时间稳健性搜索

Rescue
→ raw representation 未必强
→ 但存在非线性 / Tree-LR disagreement / interaction / boundary evidence
→ 值得获得一次 representation-search budget
```

本次 NB10 的最终推荐会直接作为 NB11 的起点。如果 Notebook 10 已经生成正式 handoff 文件，NB11 优先读取正式文件；如果没有手动 freeze，NB11 会读取 NB10 的 recommendation 与 rescue preview，并重建同一个 source pool。

对于 boundary candidate，这个重建是确定的：

```text
某条 NB9 ranking
        ↓
取前 k 个 features
        ↓
boundary_n{k}
```

所以 NB11 不会因为缺少手动 freeze 文件而重新搜索 NB10，也不会偷偷退回全部约 1,209 个原始候选。

这一本仍然遵循：

> **Compress → Expand → Screen → Validate → Freeze**

<details>
<summary><strong>📘 最小例子：为什么 NB10 Core 不是 LR 的最终 features？</strong></summary>

假设 NB10 只看 raw representation 时：

```text
X1 rank = 780
```

所以 X1 没进 Core。

但 Rescue 发现：

```text
MI 很高
Tree rank 很高
LR raw rank 很低
```

这说明一种可能性是：

```text
X1 与风险有关
但关系不是一条直线
```

那么 NB11 可以测试：

```text
raw X1
vs
raw X1 + spline(X1)
```

如果 spline 没有稳定增量价值，X1 就真正退出；如果有稳定增量价值，它才进入最终 LR representation。

</details>


## 2. 受约束的 Representation Candidate Space

如果有 `p` 个 numeric source features，仅二阶 pairwise interaction 就大约有：

```text
p × (p - 1) / 2
```

当 `p = 400` 时已经接近 80,000 对；如果再乘上 log、bin、spline、ratio、product 等 operator，搜索空间会迅速失控。

因此 Notebook 11 不做：

```text
所有 feature × 所有 operator × 所有 feature
```

而把表示搜索分成两层。

### 2.1 Local Transformation Grammar

| Source evidence / type | 优先测试的 representation |
|---|---|
| 高偏度 numeric | signed-log / winsor |
| nonlinear rescue | quantile bins / spline |
| Tree–LR disagreement | spline / bins；必要时 interaction |
| 同语义、同量纲 pair | ratio / relative change |
| NB8 tree-path pair evidence | restricted product interaction |
| categorical / boolean | 延续 NB7 frozen One-Hot 规则 |

Local branch 的基本单位是：

```text
一个 source
或
一对 sources
```

所以可以采用：

```text
candidate proposal
→ Cheap Probe
→ survivor screening
→ package evaluation
```

### 2.2 Global Representation Challenger：PCA

PCA 的基本单位不是一个 source，而是一整块 numeric design：

```text
numeric Core columns
        ↓
PC1, PC2, ..., PCK
```

本册只测试少量、预先规定的 component 数量：

```text
PCA-50
PCA-100
```

并且只压缩 numeric Core；NB7 的 missing indicators 与 categorical One-Hot 仍然保留在旁边。

PCA 最大化的是输入 `X` 的 variance，而项目真正关心的是：

```text
对 default risk 的预测价值
+
时间稳定性
```

所以：

```text
high explained variance
≠
high predictive value
```

PCA 只是一个 package-level challenger，不会替代 NB8–10 已经完成的 supervised / model-aware source search。


In [2]:
#@title 第 0 步：环境、路径与搜索预算
from pathlib import Path
from collections import defaultdict
import gc
import hashlib
import importlib.util
import json
import math
import re
import shutil
import subprocess
import sys
import time
import zipfile

# 先确认当前 Colab 运行时具备加速设备；如果没有，直接停止，避免长时间误跑。
_runtime_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if _runtime_check.returncode != 0 or not _runtime_check.stdout.strip():
    raise RuntimeError("请先在 Colab 中选择 GPU Runtime，再运行 Notebook 11。")

# 运行加速组件必须在 sklearn 导入前启用。
if importlib.util.find_spec("cuml") is None:
    print("首次运行：正在安装运行加速组件……")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--extra-index-url=https://pypi.nvidia.com",
        "cuml-cu12==26.8.0",
    ])

import cuml
from cuml.accel import install as _install_accel
from cuml.accel import is_proxy as _is_accel_proxy
_install_accel(log_level="warn")

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import sparse

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, SplineTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from IPython.display import display

ID_COL = "case_id"
TARGET_COL = "target"
WEEK_COL = "WEEK_NUM"
RANDOM_SEED = 42

DEVELOPMENT_WEEK_START = 0
DEVELOPMENT_WEEK_END = 63
PREPROCESSING_POLICY = "nb7_full_development_frozen"
PREPROCESSING_FIT_WEEK_START = 0
PREPROCESSING_FIT_WEEK_END = 63
REDUCER_FIT_POLICY = "current_training_window_only"
EVALUATION_ROLE = "development_representation_search"

STAGE1_TRAIN_START = 0
STAGE1_TRAIN_END = 55
STAGE1_ASSESS_START = 56
STAGE1_ASSESS_END = 63

TEMPORAL_FOLDS = [
    ("F1", 0, 31, 32, 39),
    ("F2", 0, 39, 40, 47),
    ("F3", 0, 47, 48, 55),
    ("F4", 0, 55, 56, 63),
]


def assert_development_slice(week_start, week_end, label="开发期切片"):
    week_start = int(week_start)
    week_end = int(week_end)
    if week_start > week_end:
        raise ValueError(f"{label}: 起始周大于结束周")
    if week_start < DEVELOPMENT_WEEK_START or week_end > DEVELOPMENT_WEEK_END:
        raise ValueError(
            f"{label}: 只能读取 Week {DEVELOPMENT_WEEK_START}–{DEVELOPMENT_WEEK_END}，"
            f"当前请求为 {week_start}–{week_end}。"
        )


def assert_temporal_split(train_start, train_end, assess_start, assess_end, label="时间切分"):
    assert_development_slice(train_start, train_end, label + " 训练")
    assert_development_slice(assess_start, assess_end, label + " 评估")
    if int(train_end) >= int(assess_start):
        raise ValueError(f"{label}: 训练期必须早于评估期")


assert_temporal_split(0, 55, 56, 63, "Stage 3")
for _name, _a, _b, _c, _d in TEMPORAL_FOLDS:
    assert_temporal_split(_a, _b, _c, _d, _name)

# 搜索预算：保持足够多样性，但不让候选空间无限扩张。
PROFILE_SAMPLE_MAX_ROWS = 60_000
PROBE_TRAIN_MAX_ROWS = 50_000
PROBE_ASSESS_MAX_ROWS = 30_000

MAX_LOG_CANDIDATES = 40
MAX_WINSOR_CANDIDATES = 30
MAX_BIN_SOURCES = 16
MAX_SPLINE_SOURCES = 10
QUANTILE_BIN_OPTIONS = [5, 8]
SPLINE_KNOT_OPTIONS = [4, 6]
SPLINE_DEGREE = 2

MAX_SEMANTIC_PAIR_GROUP_SIZE = 10
MAX_ARITHMETIC_PAIR_CANDIDATES = 40
MAX_TREE_INTERACTION_PAIR_CANDIDATES = 50

MAX_PROBE_SURVIVORS_PER_FAMILY = 15
PROBE_MIN_DELTA_GINI = 0.0

UNARY_PACKAGE_SIZES = [10, 20, 30, 40]
PAIR_PACKAGE_SIZES = [10, 20]

MAX_FULL_TEMPORAL_PACKAGES = 8

DR_COMPONENT_OPTIONS = [25, 50, 70, 85, 100]
DR_FIT_SAMPLE_MAX_ROWS = 50_000
ENABLE_GLOBAL_DR_BASELINES = True

# NB11 只比较 representation，固定一个 learner；最终模型调参留给 NB12。
LR_SEARCH_C = 1.0
LR_SEARCH_MAX_ITER = 1000
LR_SEARCH_TOL = 1e-4
REPRESENTATION_EVALUATOR_VERSION = "nb11_representation_v8_convergence"

# Run All 可以完成搜索；最后 Freeze 仍由人工选择。
RUN_SOURCE_PROFILE = True
RUN_CHEAP_PROBE = True
RUN_PACKAGE_STAGE1 = True
RUN_FULL_TEMPORAL = True
FREEZE_FINAL_LR_DESIGN = True
FINAL_LR_PACKAGE = "unary_top20__pair_top10"

USE_LOCAL_TRAIN_CACHE = True
LOCAL_TRAIN_CACHE_DIR = Path("/content/homecredit_nb11_cache")

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Notebook 11 当前按 Google Colab + Google Drive 配置。") from exc

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/HomeCredit_Project")
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
NB8_ZIP = ARTIFACT_ROOT / "nb08" / "nb08_feature_evidence_outputs.zip"
NB10_OUTPUT_DIR = ARTIFACT_ROOT / "nb10" / "final"
NB11_OUTPUT_DIR = ARTIFACT_ROOT / "nb11" / "final"
NB11_CHECKPOINT_DIR = ARTIFACT_ROOT / "nb11" / "checkpoints_v7"

NB11_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NB11_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# 快速自检：确保两个主要 estimator 已被加速层接管。
_test_lr = LogisticRegression(solver="lbfgs", penalty="l2", C=1.0, max_iter=20)
_test_pca = PCA(n_components=2)
if not _is_accel_proxy(_test_lr) or not _is_accel_proxy(_test_pca):
    raise RuntimeError("运行加速没有正确启用，请不要继续长时间运行。")

_X_test = np.random.default_rng(42).normal(size=(2000, 20)).astype(np.float32)
_y_test = (_X_test[:, 0] + 0.3 * _X_test[:, 1] > 0).astype(np.int32)
_test_lr.fit(_X_test, _y_test)
_test_pca.fit(_X_test)
del _test_lr, _test_pca, _X_test, _y_test

print("运行环境检查通过。")
print("NB10 输入：", NB10_OUTPUT_DIR)
print("NB11 checkpoint：", NB11_CHECKPOINT_DIR)
print("NB11 最终输出：", NB11_OUTPUT_DIR)


Mounted at /content/drive
运行环境检查通过。
NB10 输入： /content/drive/MyDrive/HomeCredit_Project/artifacts/nb10/final
NB11 checkpoint： /content/drive/MyDrive/HomeCredit_Project/artifacts/nb11/checkpoints_v7
NB11 最终输出： /content/drive/MyDrive/HomeCredit_Project/artifacts/nb11/final


## 3. 上游接力：NB11 使用 NB10 已经选出的 LR Source Space

Notebook 11 首先读取 Notebook 10 的最终 recommendation。

当前逻辑是：

```text
NB10 recommendation
        ↓
确定 LR Core subset
        ↓
读取 / 重建 Core features
        ↓
合并 NB10 rescue preview
        ↓
形成 NB11 representation-search source pool
```

如果 Notebook 10 已经保存下面这些正式 handoff 文件：

```text
lr_core_source_features.json
lr_transformation_rescue_features.json
lr_fe_source_pool.json
lr_nb11_handoff.json
```

则直接使用它们。

如果手动 freeze 没有打开，Notebook 11 会改用：

```text
nb10_recommendations.json
lr_source_rescue_preview.json
NB9 lr_rankings.json
```

得到同一个推荐 Core 与 Rescue Pool。

第二类上游来自 Notebook 7：

```text
numeric_preprocessing_stats.csv
categorical_encoding_rules.json
nb7_candidate_features.csv
```

它们定义 LR 已经冻结的 preprocessing contract。

因此本册不会重新决定“哪些 raw features 应该进入 source space”；它只在这个已经被 NB10 限定的边界内做 representation search。

PCA 也只作用在 NB10 推荐的 LR Core 上，不会借降维之名把全部旧候选重新放回来。


In [3]:
#@title 第 1 步：读取 NB10 推荐结果、上游文件与开发期数据

def assert_required_files(directory, filenames, label):
    directory = Path(directory)
    missing = [name for name in filenames if not (directory / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"{label} 缺少文件：{missing}\n当前目录：{directory}"
        )


def find_required_dir(required_files, preferred_candidates, label):
    required_files = set(required_files)
    for candidate in preferred_candidates:
        candidate = Path(candidate)
        if not candidate.exists():
            continue
        try:
            names = {p.name for p in candidate.iterdir()}
        except OSError:
            continue
        if required_files.issubset(names):
            return candidate

    first_required = sorted(required_files)[0]
    for first_file in PROJECT_ROOT.rglob(first_required):
        parent = first_file.parent
        try:
            names = {p.name for p in parent.iterdir()}
        except OSError:
            continue
        if required_files.issubset(names):
            return parent

    raise FileNotFoundError(f"找不到 {label}：{sorted(required_files)}")


if not NB8_ZIP.exists():
    raise FileNotFoundError(f"找不到 Notebook 8 evidence ZIP：{NB8_ZIP}")

NB7_OUTPUT_DIR = find_required_dir(
    required_files={
        "numeric_preprocessing_stats.csv",
        "categorical_encoding_rules.json",
        "nb7_candidate_features.csv",
    },
    preferred_candidates=[
        ARTIFACT_ROOT / "nb07" / "final",
        ARTIFACT_ROOT / "nb07",
        PROJECT_ROOT / "reports" / "reference_model",
    ],
    label="Notebook 7 LR preprocessing",
)

NB5_OUTPUT_DIR = find_required_dir(
    required_files={"train_clean.parquet", "feature_catalog.csv"},
    preferred_candidates=[
        ARTIFACT_ROOT / "nb05" / "final",
        ARTIFACT_ROOT / "nb05",
        PROJECT_ROOT / "clean_split",
    ],
    label="Notebook 5 clean Train",
)

NB9_OUTPUT_DIR = find_required_dir(
    required_files={
        "lr_rankings.json",
        "tree_rankings.json",
        "nb9_ranking_config.json",
    },
    preferred_candidates=[
        ARTIFACT_ROOT / "nb09" / "final",
        ARTIFACT_ROOT / "nb09",
    ],
    label="Notebook 9 rankings",
)

# ------------------------------------------------------------
# 读取 NB9 ranking 与 NB10 最终 recommendation
# ------------------------------------------------------------
with open(NB9_OUTPUT_DIR / "lr_rankings.json", encoding="utf-8") as f:
    lr_routes = json.load(f)
with open(NB9_OUTPUT_DIR / "tree_rankings.json", encoding="utf-8") as f:
    tree_routes = json.load(f)

recommendation_path = NB10_OUTPUT_DIR / "nb10_recommendations.json"
if not recommendation_path.exists():
    raise FileNotFoundError(
        "找不到 NB10 的 nb10_recommendations.json。"
        "Notebook 10 最后的保存 cell 必须至少执行一次。"
    )

with open(recommendation_path, encoding="utf-8") as f:
    nb10_recommendations = json.load(f)

lr_recommendation = nb10_recommendations.get("lr")
tree_recommendation = nb10_recommendations.get("tree")
lr_guardrail = nb10_recommendations.get("lr_freeze_guardrail", {})

if not lr_recommendation:
    raise RuntimeError("NB10 没有形成 LR recommendation，不能开始 NB11。")

if lr_guardrail.get("freeze_status") not in {None, "READY_FOR_MANUAL_FREEZE_REVIEW"}:
    raise RuntimeError(
        "NB10 的 LR recommendation 仍有未解决 guardrail："
        f"{lr_guardrail.get('required_followup')}"
    )

# ------------------------------------------------------------
# 读取 NB8 / NB7 evidence 与 preprocessing
# ------------------------------------------------------------
NB8_MEMBERS = {
    "feature_evidence": "reports/feature_evidence/feature_evidence_report.csv",
    "interaction_pairs": "reports/feature_evidence/interaction_pair_evidence.csv",
}


def read_csv_from_zip(zip_path, internal_path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(internal_path) as f:
            return pd.read_csv(f)


feature_evidence = read_csv_from_zip(NB8_ZIP, NB8_MEMBERS["feature_evidence"])
interaction_pair_evidence = read_csv_from_zip(NB8_ZIP, NB8_MEMBERS["interaction_pairs"])

numeric_stats = pd.read_csv(NB7_OUTPUT_DIR / "numeric_preprocessing_stats.csv")
nb7_catalog = pd.read_csv(NB7_OUTPUT_DIR / "nb7_candidate_features.csv")
with open(NB7_OUTPUT_DIR / "categorical_encoding_rules.json", encoding="utf-8") as f:
    lr_category_lists = json.load(f)

# ------------------------------------------------------------
# 读取正式 handoff；若未手动 freeze，则从 recommendation 精确重建
# ------------------------------------------------------------
formal_handoff_files = [
    NB10_OUTPUT_DIR / "lr_core_source_features.json",
    NB10_OUTPUT_DIR / "lr_transformation_rescue_features.json",
    NB10_OUTPUT_DIR / "lr_fe_source_pool.json",
    NB10_OUTPUT_DIR / "lr_nb11_handoff.json",
]

formal_handoff_available = all(path.exists() for path in formal_handoff_files)
source_signature_paths = []

if formal_handoff_available:
    with open(formal_handoff_files[0], encoding="utf-8") as f:
        lr_core_features = json.load(f)
    with open(formal_handoff_files[1], encoding="utf-8") as f:
        lr_rescue_features = json.load(f)
    with open(formal_handoff_files[2], encoding="utf-8") as f:
        lr_fe_source_pool = json.load(f)
    with open(formal_handoff_files[3], encoding="utf-8") as f:
        nb10_handoff = json.load(f)

    rescue_evidence_path = NB10_OUTPUT_DIR / "lr_transformation_rescue_evidence.csv"
    rescue_summary_path = NB10_OUTPUT_DIR / "lr_transformation_rescue_summary.csv"
    HANDOFF_MODE = "formal_freeze"
    source_signature_paths = formal_handoff_files

else:
    # NB10 boundary / depth candidates 都是某条 NB9 ranking 的前 k 个 features。
    # 因此 recommendation 中的 route + feature_count 足以精确重建 Core。
    lr_core_subset = str(lr_recommendation["subset"])
    lr_core_route = str(lr_recommendation["route"])
    lr_core_k = int(lr_recommendation["feature_count"])

    if lr_core_route not in lr_routes:
        raise KeyError(f"NB9 中找不到 LR route：{lr_core_route}")

    lr_core_features = list(lr_routes[lr_core_route][:lr_core_k])
    if len(lr_core_features) != lr_core_k:
        raise RuntimeError(
            f"无法重建 {lr_core_subset}：预期 {lr_core_k} 个 features，"
            f"实际得到 {len(lr_core_features)} 个。"
        )

    preview_json = NB10_OUTPUT_DIR / "lr_source_rescue_preview.json"
    preview_summary = NB10_OUTPUT_DIR / "lr_source_rescue_preview_summary.csv"
    preview_evidence = NB10_OUTPUT_DIR / "lr_source_rescue_preview_evidence.csv"

    if preview_json.exists():
        with open(preview_json, encoding="utf-8") as f:
            rescue_preview = json.load(f)
        lr_rescue_features = list(rescue_preview.get("rescue_features", []))
    elif preview_summary.exists():
        _preview_summary = pd.read_csv(preview_summary)
        if "feature" not in _preview_summary.columns:
            raise RuntimeError("LR rescue preview summary 缺少 feature 列。")
        lr_rescue_features = _preview_summary["feature"].dropna().astype(str).tolist()
    else:
        raise FileNotFoundError(
            "NB10 没有正式 freeze 文件，也没有 lr_source_rescue_preview。\n"
            "当前 Notebook 10 的搜索结果本身是完整的，但 NB11 还需要 Rescue Pool。\n"
            "只需在 NB10 运行 13.5 LR Source-Space Rescue 与最后保存 cell，"
            "不需要重新训练 Stage 1 或 F1–F4。"
        )

    lr_fe_source_pool = list(dict.fromkeys([*lr_core_features, *lr_rescue_features]))

    nb10_handoff = {
        "lr_core_subset": lr_core_subset,
        "lr_core_route": lr_core_route,
        "core_feature_count": len(lr_core_features),
        "rescue_feature_count": len(lr_rescue_features),
        "fe_source_pool_count": len(lr_fe_source_pool),
        "role": "NB10 recommendation + rescue preview handoff for NB11",
        "validation_read": False,
        "oot_read": False,
        "handoff_mode": "recommendation_preview",
    }

    rescue_evidence_path = preview_evidence
    rescue_summary_path = preview_summary
    HANDOFF_MODE = "recommendation_preview"
    source_signature_paths = [
        recommendation_path,
        NB9_OUTPUT_DIR / "lr_rankings.json",
    ]
    if preview_json.exists():
        source_signature_paths.append(preview_json)
    elif preview_summary.exists():
        source_signature_paths.append(preview_summary)

rescue_evidence = (
    pd.read_csv(rescue_evidence_path)
    if rescue_evidence_path.exists()
    else pd.DataFrame()
)
rescue_summary = (
    pd.read_csv(rescue_summary_path)
    if rescue_summary_path.exists()
    else pd.DataFrame()
)

# ------------------------------------------------------------
# 开发期数据：复制到本地后再做长循环
# ------------------------------------------------------------
SOURCE_TRAIN_PATH = NB5_OUTPUT_DIR / "train_clean.parquet"
TRAIN_PATH = SOURCE_TRAIN_PATH

if USE_LOCAL_TRAIN_CACHE:
    LOCAL_TRAIN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    local_train_path = LOCAL_TRAIN_CACHE_DIR / "train_clean.parquet"
    source_size = SOURCE_TRAIN_PATH.stat().st_size
    local_ok = local_train_path.exists() and local_train_path.stat().st_size == source_size
    if not local_ok:
        print("正在把训练数据复制到当前运行环境的本地磁盘（只做一次）……")
        shutil.copy2(SOURCE_TRAIN_PATH, local_train_path)
    TRAIN_PATH = local_train_path

train_parquet = pq.ParquetFile(TRAIN_PATH)
train_meta = pd.read_parquet(
    TRAIN_PATH,
    columns=[ID_COL, WEEK_COL, TARGET_COL],
    engine="pyarrow",
).reset_index(drop=True)

train_week_min = int(train_meta[WEEK_COL].min())
train_week_max = int(train_meta[WEEK_COL].max())
if train_week_min < DEVELOPMENT_WEEK_START or train_week_max > DEVELOPMENT_WEEK_END:
    raise RuntimeError(
        f"开发期数据边界异常：观察到 Week {train_week_min}–{train_week_max}。"
    )

# WEEK_NUM 与 row-group 位置固定，后续所有时间窗口复用同一份索引。
ROW_GROUP_WEEK_CACHE = [
    train_parquet.read_row_group(rg, columns=[WEEK_COL], use_threads=True)
    .column(0)
    .to_numpy(zero_copy_only=False)
    for rg in range(train_parquet.num_row_groups)
]

ROW_GROUP_ROW_COUNTS = [
    int(train_parquet.metadata.row_group(rg).num_rows)
    for rg in range(train_parquet.num_row_groups)
]
ROW_GROUP_OFFSETS = np.cumsum([0, *ROW_GROUP_ROW_COUNTS[:-1]]).astype(np.int64)

WINDOW_ROW_INDEX_CACHE = {}
WINDOW_META_CACHE = {}


def get_window_row_groups(week_start, week_end):
    assert_development_slice(week_start, week_end, label="时间窗口")
    key = (int(week_start), int(week_end))
    if key not in WINDOW_ROW_INDEX_CACHE:
        selected = []
        for rg, weeks in enumerate(ROW_GROUP_WEEK_CACHE):
            local_indices = np.flatnonzero(
                (weeks >= week_start) & (weeks <= week_end)
            ).astype(np.int32)
            if local_indices.size:
                selected.append((rg, local_indices))
        WINDOW_ROW_INDEX_CACHE[key] = selected
    return WINDOW_ROW_INDEX_CACHE[key]


def get_window_meta(week_start, week_end):
    key = (int(week_start), int(week_end))
    if key not in WINDOW_META_CACHE:
        parts = []
        for rg, local_indices in get_window_row_groups(week_start, week_end):
            global_indices = ROW_GROUP_OFFSETS[rg] + local_indices.astype(np.int64)
            parts.append(train_meta.iloc[global_indices][[ID_COL, WEEK_COL, TARGET_COL]].copy())
        WINDOW_META_CACHE[key] = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
            columns=[ID_COL, WEEK_COL, TARGET_COL]
        )
    return WINDOW_META_CACHE[key]


def sha256_file(path, chunk_bytes=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_bytes)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def combined_file_signature(paths):
    digest = hashlib.sha256()
    for path in sorted([Path(path) for path in paths], key=lambda p: str(p)):
        digest.update(path.name.encode("utf-8"))
        digest.update(sha256_file(path).encode("utf-8"))
    return digest.hexdigest()[:20]


NB7_PREPROCESSING_SIGNATURE = combined_file_signature([
    NB7_OUTPUT_DIR / "numeric_preprocessing_stats.csv",
    NB7_OUTPUT_DIR / "categorical_encoding_rules.json",
    NB7_OUTPUT_DIR / "nb7_candidate_features.csv",
])
NB10_SOURCE_SIGNATURE = combined_file_signature(source_signature_paths)

schema_columns = set(train_parquet.schema.names)
unknown = sorted(set(lr_fe_source_pool) - schema_columns)
if unknown:
    raise ValueError(f"NB10 handoff 中存在训练数据不认识的 features：{unknown[:20]}")
if not set(lr_core_features).issubset(set(lr_fe_source_pool)):
    raise ValueError("LR Core 必须是 lr_fe_source_pool 的子集。")

print("NB10 推荐 LR subset：", lr_recommendation["subset"])
print("NB10 推荐 LR route：", lr_recommendation["route"])
print("Handoff 模式：", HANDOFF_MODE)
print("LR Core 特征数：", len(lr_core_features))
print("LR Rescue 特征数：", len(lr_rescue_features))
print("NB11 候选 Source 总数：", len(lr_fe_source_pool))
print("开发期 Week：", f"{train_week_min}–{train_week_max}")
print("是否读取 Validation / OOT：", False)


正在把训练数据复制到当前运行环境的本地磁盘（只做一次）……
NB10 推荐 LR subset： lr__redundancy_aware__boundary_n100
NB10 推荐 LR route： redundancy_aware
Handoff 模式： recommendation_preview
LR Core 特征数： 100
LR Rescue 特征数： 120
NB11 候选 Source 总数： 220
开发期 Week： 0–63
是否读取 Validation / OOT： False


## 4. Source Audit 与变换候选分配

Representation search 的第一笔成本不是训练模型，而是判断：

```text
这个 source feature
到底允许进入哪些 operator family？
```

例如：

```text
高度偏斜金额
→ log / winsor 值得测试

只有 0/1 的 numeric
→ spline 没有意义

nonlinear rescue
→ bins / spline 优先级更高
```

所以先建立一个 `source_profile`：

```text
dtype
missing rate
unique count
skewness
quantiles
Core / Rescue 身份
Rescue channel
```

然后 Transformation Grammar 根据 profile 分配预算。

这一步仍然不训练 LR。

<details>
<summary><strong>📘 最小例子：为什么先判断“可不可以变”，再决定“变成什么”？</strong></summary>

假设：

```text
X = {0, 1}
```

搜索：

```text
log(X)
sqrt(X)
spline(X)
```

几乎没有新的表示能力，却增加候选数量。

相反，一个：

```text
income ∈ [0, 500000]
右偏很严重
```

测试：

```text
raw income
signed_log1p(income)
winsor(income)
```

才可能真正改变 LR 的几何表示。

</details>


In [4]:
#@title 第 2 步：Source Audit（辅助代码，可折叠）
# ============================================================
# 3. Source profiling
# ============================================================

def _build_sample_selector(
    week_start,
    week_end,
    max_rows,
    seed,
):
    """
    只根据已经缓存的 WEEK_NUM 选择抽样行；这一阶段不读取 feature columns。
    """
    key = (
        int(week_start),
        int(week_end),
        int(max_rows),
        int(seed),
    )

    if key in SAMPLE_SELECTOR_CACHE:
        return SAMPLE_SELECTOR_CACHE[key]

    rng = np.random.default_rng(seed)

    res_keys = np.empty(0, dtype=np.float64)
    res_rgs = np.empty(0, dtype=np.int32)
    res_local = np.empty(0, dtype=np.int32)

    for rg, local_indices in get_window_row_groups(
        week_start,
        week_end,
    ):
        keys = rng.random(local_indices.size)

        if local_indices.size > max_rows:
            keep = np.argpartition(keys, max_rows - 1)[:max_rows]
            keys = keys[keep]
            local_indices = local_indices[keep]

        rg_values = np.full(
            local_indices.size,
            int(rg),
            dtype=np.int32,
        )

        res_keys = np.concatenate([res_keys, keys])
        res_rgs = np.concatenate([res_rgs, rg_values])
        res_local = np.concatenate([res_local, local_indices])

        if res_keys.size > max_rows:
            keep = np.argpartition(res_keys, max_rows - 1)[:max_rows]
            res_keys = res_keys[keep]
            res_rgs = res_rgs[keep]
            res_local = res_local[keep]

    if res_keys.size:
        order = np.argsort(res_keys, kind="stable")
        selector = pd.DataFrame({
            "row_group": res_rgs[order],
            "local_index": res_local[order],
            "__sample_key__": res_keys[order],
        })
    else:
        selector = pd.DataFrame(
            columns=["row_group", "local_index", "__sample_key__"]
        )

    SAMPLE_SELECTOR_CACHE[key] = selector
    return selector


SAMPLE_SELECTOR_CACHE = {}


def sample_window(columns, week_start, week_end, max_rows, seed=RANDOM_SEED):
    """
    从开发期窗口中做确定性抽样；只读取最终被抽中的 feature rows。
    """
    assert_development_slice(
        week_start,
        week_end,
        label="sample_window",
    )

    wanted = list(dict.fromkeys([WEEK_COL, TARGET_COL, *columns]))
    selector = _build_sample_selector(
        week_start,
        week_end,
        max_rows,
        seed,
    )

    if selector.empty:
        return pd.DataFrame(columns=wanted)

    parts = []

    for rg, part in selector.groupby("row_group", sort=True):
        local_indices = part["local_index"].to_numpy(dtype=np.int32)

        table = train_parquet.read_row_group(
            int(rg),
            columns=wanted,
            use_threads=True,
        )
        table = table.take(pa.array(local_indices))

        frame = table.to_pandas()
        frame["__sample_key__"] = part["__sample_key__"].to_numpy()
        parts.append(frame)

    return (
        pd.concat(parts, ignore_index=True)
        .sort_values("__sample_key__", kind="stable")
        .drop(columns="__sample_key__")
        .reset_index(drop=True)
    )


dtype_lookup = nb7_catalog.set_index("feature")["dtype_family"].to_dict()
feature_evidence_indexed = feature_evidence.drop_duplicates("feature").set_index("feature")

rescue_channel_map = defaultdict(set)
if not rescue_evidence.empty and {"feature", "rescue_channel"}.issubset(rescue_evidence.columns):
    for row in rescue_evidence[["feature", "rescue_channel"]].dropna().itertuples(index=False):
        rescue_channel_map[row.feature].add(row.rescue_channel)


def build_source_profile():
    numeric_sources = [
        f for f in lr_fe_source_pool
        if dtype_lookup.get(f) == "numeric"
    ]

    sample = sample_window(
        numeric_sources,
        STAGE1_TRAIN_START,
        STAGE1_TRAIN_END,
        PROFILE_SAMPLE_MAX_ROWS,
    )

    rows = []
    for feature in lr_fe_source_pool:
        dtype_family = dtype_lookup.get(feature, "unknown")
        row = {
            "feature": feature,
            "dtype_family": dtype_family,
            "in_core": feature in set(lr_core_features),
            "in_rescue": feature in set(lr_rescue_features),
            "rescue_channels": "|".join(sorted(rescue_channel_map.get(feature, set()))),
        }

        if feature in feature_evidence_indexed.index:
            ev = feature_evidence_indexed.loc[feature]
            for col in ["iv_percentile", "mi_percentile", "gain_percentile", "permutation_percentile"]:
                if col in ev.index:
                    row[col] = pd.to_numeric(ev[col], errors="coerce")

        if dtype_family == "numeric" and feature in sample.columns:
            x = pd.to_numeric(sample[feature], errors="coerce")
            finite = x[np.isfinite(x)]
            row.update({
                "sample_missing_rate": float(x.isna().mean()),
                "sample_unique": int(finite.nunique()),
                "sample_skew": float(finite.skew()) if len(finite) > 2 else np.nan,
                "q01": float(finite.quantile(0.01)) if len(finite) else np.nan,
                "q50": float(finite.quantile(0.50)) if len(finite) else np.nan,
                "q99": float(finite.quantile(0.99)) if len(finite) else np.nan,
            })
        rows.append(row)

    return pd.DataFrame(rows)


SOURCE_PROFILE_PATH = NB11_CHECKPOINT_DIR / f"source_profile_{REPRESENTATION_EVALUATOR_VERSION}.csv"
source_profile = pd.DataFrame()

if RUN_SOURCE_PROFILE:
    source_profile = build_source_profile()
    source_profile.to_csv(SOURCE_PROFILE_PATH, index=False)
elif SOURCE_PROFILE_PATH.exists():
    source_profile = pd.read_csv(SOURCE_PROFILE_PATH)
else:
    print("Source Profile 尚未生成。")

if not source_profile.empty:
    display(source_profile.head(30))


,feature,dtype_family,in_core,in_rescue,rescue_channels,iv_percentile,mi_percentile,gain_percentile,permutation_percentile,sample_missing_rate,sample_unique,sample_skew,q01,q50,q99
0,datelastinstal40dpd_247D__days_before_decision,numeric,True,False,,0.948718,0.848635,0.891646,0.220000,0.925350,2688.0,0.482726,47.000000,1531.000000,4747.880000
1,pmtcount_693L,numeric,True,False,,0.830438,0.938792,0.275434,NaN,0.870750,23.0,0.761923,0.000000,6.000000,12.000000
2,overdueamountmax2date_1142D_min__days_before_d...,numeric,True,False,,0.946237,0.780811,0.684036,NaN,0.668350,2559.0,1.177575,-6.000000,404.000000,2625.000000
3,assignmentdate_238D__days_before_decision,numeric,True,False,,0.849462,0.874276,0.275434,NaN,0.878767,3905.0,1.345604,-14.000000,1864.500000,9518.430000
4,numberofoverdueinstlmaxdat_641D_max__days_befo...,numeric,True,False,,0.962779,0.754342,0.883375,0.340000,0.671650,2204.0,1.360242,-10.000000,218.000000,2094.000000
5,dpdmaxdateyear_596T_max,numeric,True,False,,0.934657,0.770885,0.275434,NaN,0.179883,5.0,-0.819641,2017.000000,2019.000000,2020.000000
6,lastdelinqdate_224D__days_before_decision,numeric,True,False,,0.941274,0.751034,0.966088,0.893333,0.662450,2278.0,2.245984,-11.000000,284.000000,2724.240000
7,dpdmaxdateyear_896T_max,numeric,True,False,,0.923077,0.768404,0.691481,NaN,0.414933,16.0,-2.688247,2008.000000,2018.000000,2019.000000
8,numberofoverdueinstlmaxdat_641D_min__days_befo...,numeric,True,False,,0.951199,0.739454,0.832920,NaN,0.671650,2277.0,0.971754,-7.000000,369.000000,2139.000000
9,overdueamountmax2date_1002D_max__days_before_d...,numeric,True,False,,0.937965,0.751861,0.916460,0.506667,0.592117,4074.0,1.289954,33.000000,856.000000,4361.560000


## 5. Local Transformation Grammar

本册把 Local representation 分成四个 family：

```text
A. Tail / Scale Transform
   signed_log1p
   winsor_1_99

B. Nonlinear Basis
   quantile bins: 5 / 8
   spline knots: 4 / 6

C. Business-like Relative Arithmetic
   ratio
   relative_change

D. Restricted Interaction
   product
```

这里有两个不同层次的搜索：

```text
Operator Search
→ log 还是 spline？

Operator-Parameter Search
→ spline 用 4 knots 还是 6 knots？
```

它们属于 **representation search**，不是 Notebook 12 的模型超参数搜索。

Notebook 12 才处理：

```text
LR regularization strength
penalty
solver / convergence
```

这样可以避免过早形成：

```text
Feature Representation × Model Hyperparameters
```

的笛卡尔积。

### 为什么不直接对全部特征做 `PolynomialFeatures(degree=2)`？

因为二阶展开会让列数随输入维度快速增长。这里借用 interaction feature 的思想，但只在已有证据的 pair 上测试 product，而不是对整个 source pool 做全量 degree-2 expansion。

### PCA 为什么不进入这些 Local families？

上面的四类都满足：

```text
一个 source / 一对 sources
→ 一个 candidate representation
```

PCA 改变的是整块 numeric Core，因此它不参与 Local candidate budgets，也不进入单候选 Cheap Probe；它会在 package comparison 阶段直接与 Raw Core、Local FE 比较。



### 5.5 `+ / - / × / ÷` 并不是四种同等有价值的 Feature Engineering

这里有一个很容易被忽略、但对 Logistic Regression 非常重要的线性代数事实。

假设最终 LR 已经同时包含：

```text
A
B
```

那么再加入：

```text
A + B
```

或者：

```text
A - B
```

通常**不会扩大模型可以表达的函数空间**。

因为：

```text
β1 A + β2 B + β3 (A - B)
=
(β1 + β3) A + (β2 - β3) B
```

它仍然只是 `A` 与 `B` 的线性组合。

所以本册默认不花搜索预算枚举：

```text
A + B
A - B
```

除非以后有非常特殊的业务约束、变量替换或正则化结构需要它们。

真正能改变普通 LR 表达能力的是：

```text
A × B          → interaction
A / B          → nonlinear relative scale
(A - B) / |B|  → relative change
bin(A)         → threshold / step effect
spline(A)      → smooth nonlinear effect
```

这也是为什么“加减乘除”不能机械地全部生成。

<details>
<summary><strong>📘 最小例子：difference 为什么常常是冗余的？</strong></summary>

原模型：

```text
logit(p) = b0 + b1 A + b2 B
```

加入：

```text
D = A - B
```

得到：

```text
logit(p) = b0 + b1 A + b2 B + b3 D
```

代入 `D = A - B`：

```text
= b0 + (b1 + b3)A + (b2 - b3)B
```

所以没有新的函数形状。

但加入：

```text
A × B
```

以后：

```text
A 对风险的影响
可以随 B 改变
```

这才真正扩大表示空间。

</details>

### 为什么本册暂时不把 WOE / Target Encoding 放进默认主线？

WOE / Target Encoding 本身不是错误方法，信用评分里也很常见。

但它们使用 `target` 来学习 representation，因此如果：

```text
先用整个 Train 的 target 算 encoding
再在同一批 Train 上训练 LR
```

会产生明显的 target leakage / overfitting 风险。

如果以后要加入，必须做：

```text
fold-local / cross-fitted encoding
+
time-aware validation
```

这会新增一整条监督式 representation search 分支。

当前已经有 NB7 frozen One-Hot baseline，而且本项目时间预算有限，所以 NB11 主线先把预算放在：

```text
numeric nonlinear representation
+
restricted relative features
+
restricted interactions
```

WOE / Target Encoding 保留为 **optional extension**，而不是现在默认扩张搜索空间。


In [5]:
#@title 第 3 步：Candidate 定义辅助函数（可折叠）
# ============================================================
# Candidate specification helpers
# ============================================================

def stable_candidate_id(family, sources, params):
    payload = json.dumps(
        {"family": family, "sources": list(sources), "params": params},
        sort_keys=True,
        ensure_ascii=False,
    ).encode("utf-8")
    return f"{family}__{hashlib.sha1(payload).hexdigest()[:12]}"


def make_spec(family, sources, params=None, reason=""):
    params = params or {}
    return {
        "candidate_id": stable_candidate_id(family, sources, params),
        "family": family,
        "sources": list(sources),
        "params": params,
        "reason": reason,
    }


def source_priority_score(row):
    # Proposal ordering only; not a predictive final score.
    values = []
    for col in ["mi_percentile", "iv_percentile", "gain_percentile", "permutation_percentile"]:
        value = pd.to_numeric(row.get(col, np.nan), errors="coerce")
        if pd.notna(value):
            values.append(float(value))
    base = max(values) if values else 0.0
    if bool(row.get("in_rescue", False)):
        base += 0.05
    return base


def semantic_stem(name):
    """A conservative name heuristic for same-concept aggregate pairs."""
    text = str(name).lower()
    text = re.sub(r"\d+", "", text)
    tokens = [t for t in re.split(r"[^a-z]+", text) if t]
    drop = {
        "mean", "avg", "min", "max", "sum", "std", "var", "median",
        "first", "last", "count", "cnt", "latest", "recent",
    }
    kept = [t for t in tokens if t not in drop]
    return "_".join(kept[:5])


def estimate_added_columns(spec):
    family = spec["family"]
    if family == "qbin":
        return int(spec["params"]["n_bins"])
    if family == "spline":
        # Exact number is learned by SplineTransformer; this is a budget estimate.
        return int(spec["params"]["n_knots"] + SPLINE_DEGREE - 1)
    return 1


In [6]:
#@title Candidate Space Construction
#@title 第 4 步：生成 Unary + Pair 候选空间
# ============================================================
# Build Unary + Pair proposal space
# ============================================================

def build_candidate_space(source_profile):
    if source_profile.empty:
        raise RuntimeError("先运行 Source Profile。")

    numeric = source_profile.loc[source_profile["dtype_family"].eq("numeric")].copy()
    numeric["priority"] = numeric.apply(source_priority_score, axis=1)

    specs = []

    # --------------------------------------------------------
    # A. Tail / scale transforms
    # --------------------------------------------------------
    log_pool = numeric.loc[
        numeric["sample_unique"].fillna(0).gt(10)
        & numeric["sample_skew"].abs().fillna(0).ge(1.5)
    ].sort_values(["priority", "sample_skew"], ascending=[False, False]).head(MAX_LOG_CANDIDATES)

    for row in log_pool.itertuples(index=False):
        specs.append(make_spec(
            "signed_log1p", [row.feature],
            reason="High-skew numeric source; test a monotone scale compression.",
        ))

    winsor_pool = numeric.loc[
        numeric["sample_unique"].fillna(0).gt(10)
    ].copy()
    winsor_pool["tail_span"] = (
        pd.to_numeric(winsor_pool["q99"], errors="coerce")
        - pd.to_numeric(winsor_pool["q01"], errors="coerce")
    ).abs()
    winsor_pool = winsor_pool.sort_values(["priority", "tail_span"], ascending=[False, False]).head(MAX_WINSOR_CANDIDATES)

    for row in winsor_pool.itertuples(index=False):
        specs.append(make_spec(
            "winsor", [row.feature], {"lower_q": 0.01, "upper_q": 0.99},
            reason="Tail-sensitive numeric source; test robust clipping.",
        ))

    # --------------------------------------------------------
    # B. Nonlinear bases
    # Prefer structural rescue signals instead of all numerics.
    # --------------------------------------------------------
    nonlinear_mask = numeric["rescue_channels"].fillna("").str.contains(
        "nonlinear_disagreement|tree_lr_disagreement", regex=True
    )
    nonlinear_pool = numeric.loc[nonlinear_mask].sort_values("priority", ascending=False)

    bin_pool = nonlinear_pool.head(MAX_BIN_SOURCES)
    spline_pool = nonlinear_pool.head(MAX_SPLINE_SOURCES)

    for row in bin_pool.itertuples(index=False):
        for n_bins in QUANTILE_BIN_OPTIONS:
            specs.append(make_spec(
                "qbin", [row.feature], {"n_bins": int(n_bins)},
                reason="Nonlinear rescue source; quantile bins can express threshold/step effects.",
            ))

    for row in spline_pool.itertuples(index=False):
        for n_knots in SPLINE_KNOT_OPTIONS:
            specs.append(make_spec(
                "spline", [row.feature], {"n_knots": int(n_knots), "degree": SPLINE_DEGREE},
                reason="Nonlinear rescue source; spline gives a controlled smooth basis.",
            ))

    # --------------------------------------------------------
    # C. Semantic arithmetic pairs
    # Only same-stem groups; no all-pairs arithmetic explosion.
    # --------------------------------------------------------
    numeric["stem"] = numeric["feature"].map(semantic_stem)
    arithmetic_rows = []
    for stem, group in numeric.groupby("stem"):
        if not stem or len(group) < 2:
            continue
        group = group.sort_values("priority", ascending=False).head(MAX_SEMANTIC_PAIR_GROUP_SIZE)
        feats = group["feature"].tolist()
        priority = dict(zip(group["feature"], group["priority"]))
        for i, a in enumerate(feats):
            for b in feats[i+1:]:
                arithmetic_rows.append((priority[a] + priority[b], a, b, stem))

    arithmetic_rows.sort(reverse=True)
    for _, a, b, stem in arithmetic_rows[:MAX_ARITHMETIC_PAIR_CANDIDATES]:
        # Difference / sum are not generated by default because,
        # when raw A and B remain in a linear model, they do not enlarge
        # the linear span. Ratio / relative change do.
        specs.append(make_spec(
            "ratio", [a, b], {"epsilon": 1e-6},
            reason=f"Same semantic stem ({stem}); test A/B relative scale.",
        ))
        specs.append(make_spec(
            "ratio", [b, a], {"epsilon": 1e-6},
            reason=f"Same semantic stem ({stem}); test B/A reverse relative scale.",
        ))
        specs.append(make_spec(
            "relative_change", [a, b], {"epsilon": 1e-6},
            reason=f"Same semantic stem ({stem}); test (A-B)/|B| relative change.",
        ))

    # --------------------------------------------------------
    # D. Tree-path interaction proposals from NB8
    # --------------------------------------------------------
    pair_table = interaction_pair_evidence.copy()
    required_pair_cols = {"feature_a", "feature_b", "shared_path_count"}
    if required_pair_cols.issubset(pair_table.columns):
        pool = set(lr_fe_source_pool)
        core_set = set(lr_core_features)
        rescue_set = set(lr_rescue_features)

        pair_table = pair_table.loc[
            pair_table["feature_a"].isin(pool)
            & pair_table["feature_b"].isin(pool)
            & pair_table["feature_a"].map(lambda x: dtype_lookup.get(x) == "numeric")
            & pair_table["feature_b"].map(lambda x: dtype_lookup.get(x) == "numeric")
        ].copy()
        pair_table["shared_path_count"] = pd.to_numeric(
            pair_table["shared_path_count"], errors="coerce"
        )

        # Weak computational hierarchy:
        # Core-Core and Core-Rescue are eligible with pair evidence.
        # Rescue-Rescue must also lie in the strongest 20% of eligible
        # shared-path evidence before receiving interaction budget.
        strong_rr_cut = (
            float(pair_table["shared_path_count"].quantile(0.80))
            if not pair_table.empty else np.inf
        )

        def pair_allowed(row):
            a_core = row.feature_a in core_set
            b_core = row.feature_b in core_set
            a_rescue = row.feature_a in rescue_set
            b_rescue = row.feature_b in rescue_set
            if a_core and b_core:
                return True
            if (a_core and b_rescue) or (b_core and a_rescue):
                return True
            if a_rescue and b_rescue:
                return row.shared_path_count >= strong_rr_cut
            return False

        pair_table = pair_table.sort_values("shared_path_count", ascending=False)
        pair_table = pair_table.loc[
            [pair_allowed(row) for row in pair_table.itertuples(index=False)]
        ].head(MAX_TREE_INTERACTION_PAIR_CANDIDATES)

        for row in pair_table.itertuples(index=False):
            specs.append(make_spec(
                "product", [row.feature_a, row.feature_b],
                params={"tree_shared_path_count": int(row.shared_path_count)},
                reason="NB8 tree-path pair evidence under weak computational hierarchy.",
            ))

    # Deduplicate exact specs.
    unique = {spec["candidate_id"]: spec for spec in specs}
    specs = list(unique.values())

    catalog = pd.DataFrame([
        {
            **spec,
            "sources_text": " | ".join(spec["sources"]),
            "params_json": json.dumps(spec["params"], ensure_ascii=False, sort_keys=True),
            "estimated_added_columns": estimate_added_columns(spec),
        }
        for spec in specs
    ])
    return specs, catalog


candidate_specs = []
candidate_catalog = pd.DataFrame()

if not source_profile.empty:
    candidate_specs, candidate_catalog = build_candidate_space(source_profile)
    candidate_catalog.to_csv(NB11_OUTPUT_DIR / "representation_candidate_catalog.csv", index=False)
    print("表示候选总数：", len(candidate_catalog))
    display(candidate_catalog.groupby("family").agg(
        candidates=("candidate_id", "count"),
        estimated_columns=("estimated_added_columns", "sum"),
    ).reset_index())


表示候选总数： 292


,family,candidates,estimated_columns
0,product,50,50
1,qbin,32,208
2,ratio,80,80
3,relative_change,40,40
4,signed_log1p,40,40
5,spline,20,120
6,winsor,30,30


## 6. 候选表示的计算成本

不同 transformation 的成本并不完全一样。

例如：

```text
signed_log1p
→ 通常只增加 1 column

spline
→ 可能增加多个 basis columns
```

PCA 则是另一种情况：

```text
Local FE
→ 通常增加 design columns

PCA
→ 压缩 numeric block
→ 最终 design columns 可能减少
```

因此本册不建立复杂的工程成本评分，只记录最容易解释的几项：

```text
Cheap Probe
- ΔGini
- added columns

Package Comparison
- final design columns
- fit seconds（辅助参考）
```

最后判断时仍然优先看：

```text
预测表现
+
时间稳定性
+
表示复杂度
```

`fit_seconds` 只是补充信息，不和 Gini 强行揉成一个没有自然单位的综合分数。


## 7. 候选评估函数与搜索空间收缩

到这里已经有了：

```text
Candidate Space
+
Search Cost
```

但还缺第三个核心组件：

```text
Evaluator
```

因为“搜索空间很大”本身不是问题。
真正的问题是：

> **每产生一个 candidate，我们凭什么决定它应该继续、暂停，还是退出？**

Notebook 11 不使用一个评价标准从头管到尾，而使用 **Evaluation Ladder**：

| 层级 | 搜索对象 | 主要评价 | 作用 | 不能代表什么 |
|---|---|---|---|---|
| L1 Cheap Probe | 单个 transformation | `ΔGini` | 低成本判断“有没有增量迹象” | 不是最终模型表现 |
| L2 Full-Core Package | representation package | overall Gini + weekly Gini + slope + residual SD + stability metric | 判断加入完整 Core 后是否仍有价值 | 仍只看一个开发时间段 |
| L3 F1–F4 Temporal | 少量 finalist packages | mean/worst fold Gini + fold SD + mean stability | 判断收益能否跨时间重复 | 仍不是 OOT |
| Final OOT | NB12–13 已冻结的完整方案 | competition-aligned temporal metrics | 最终未来检验 | 不能再拿来继续调 NB11 |

因此评价标准也会随着搜索成本升级：

```text
便宜 evaluator
→ 大量淘汰

更完整 evaluator
→ 少量候选比较

跨时间 evaluator
→ 决定能否 Freeze
```

这与搜索空间主线是同一个系统，而不是额外附加的一张表。

### 为什么 L1 先用 ΔGini，而不是一开始就用最终 stability metric？

Cheap Probe 的目标只是 screening。
它只比较：

```text
raw representation
vs
raw + transformed representation
```

所以最直接的问题是：

> **这个 transformation 有没有带来 ranking discrimination 的增量？**

本项目最终比赛指标建立在 weekly Gini 上，因此 `ΔGini` 是一个便宜、方向一致的第一层信号。

但它故意不回答：

```text
这个增益是否跨周稳定？
是否只在一个时间窗口偶然出现？
是否值得增加很多 basis columns？
```

这些问题留给更昂贵的 L2 / L3 evaluator。

### Overall Gini、Mean Weekly Gini 和 Stability Metric 不是同一个东西

```text
Overall Gini
→ 把整个 assess window 放在一起
→ 看整体排序能力

Mean Weekly Gini
→ 每周先算 Gini，再取平均
→ 尊重 WEEK_NUM 的时间粒度

Slope
→ 看 weekly Gini 是否持续向下

Residual SD
→ 看 weekly Gini 是否围绕趋势线剧烈波动

Stability Metric
→ Mean Weekly Gini
   + 对负 slope 的惩罚
   + 对 residual volatility 的惩罚
```

Home Credit 官方评价本身就是这个结构：高平均 weekly Gini 不够，时间下降和不规则波动也会被惩罚。

<details>
<summary><strong>📘 最小例子：两个 package 平均 Gini 一样，为什么评价可能完全不同？</strong></summary>

假设两个 representation package 的 weekly Gini 是：

```text
Package A
0.42, 0.42, 0.42, 0.42

Package B
0.48, 0.44, 0.40, 0.36
```

两者平均值都是：

```text
0.42
```

如果只看 mean Gini：

```text
A = B
```

但 B 有明显负 slope：

```text
时间越往后，预测能力越差
```

所以 Home Credit stability metric 会更偏向 A。

这正是为什么本项目不能把 Feature Engineering 写成：

```text
谁的 AUC/Gini 最大
→ 谁赢
```

而必须把“未来时间稳定性”放进 evaluator。

</details>

### Evaluator 和 Cost 是两个正交问题

评价标准回答：

```text
这个 representation 好不好？
```

成本回答：

```text
为了这个改善付了多少？
```

Notebook 11 不把二者强行揉成：

```text
FinalScore = Gini - 0.001 × seconds - 0.01 × columns
```

因为这些单位之间没有自然换算关系。

因此本册把两类信息分开保留：

```text
Performance / Stability
+
Complexity / Runtime
```

先看表现与稳定性；如果方案接近，再优先选择更简单的 representation。

<details>
<summary><strong>📘 最小例子：为什么评价指标和成本不能硬变成一个分数？</strong></summary>

```text
Package A
stability = 0.420
+ 5 columns
10 s

Package B
stability = 0.421
+ 80 columns
55 s
```

B 的预测评价略高，但代价大很多。

这里没有一个天然正确的公式告诉我们：

```text
+0.001 stability
到底值多少 columns / seconds？
```

所以更稳妥的做法是：

1. 先判断改善是否真实且能跨时间重复；
2. 再看是否进入 performance plateau；
3. 性能接近时优先较简单 representation。

</details>

### 为什么这里暂时不用 LogLoss / Brier 作为主评价？

Logistic Regression 可以输出概率，所以 LogLoss / Brier 当然有意义。

但 NB11 当前搜索目标是：

```text
找到能提高比赛所需 discrimination + temporal stability 的 representation
```

而不是完成最终 probability calibration。

因此：

```text
NB11
→ Gini / temporal stability 为主

NB14
→ Calibration Curve / Brier Score
```

这样避免在 Feature Engineering 阶段同时优化两个不同目标，导致“representation effect”和“calibration objective”再次纠缠。


## 8. Stage 1 — Cheap Representation Probe

如果每一个 candidate 一生成就立刻运行：

```text
完整 Core LR
×
全 Train rows
×
F1–F4
```

成本会非常高。

所以第一轮只做一个 **Cheap Probe**。

### Unary candidate

比较：

```text
raw X
vs
raw X + transformed(X)
```

### Pair candidate

比较：

```text
raw A + raw B
vs
raw A + raw B + transform(A,B)
```

这一层使用：

```text
Train 0–55
Assess 56–63
```

并使用受控样本量。

它只回答：

> **这个 transformation 是否至少显示出值得继续支付完整 Core + temporal evaluation 成本的迹象？**

因此：

```text
ΔGini < 0
→ 通常优先淘汰

ΔGini > 0
→ 只是获得进入下一轮的资格
```

不能把 Cheap Probe Top 1 直接解释成最终最好的 Feature Engineering，因为它还没有经过完整 Core context 和跨时间检查。

### PCA 不经过这一层

PCA 改变的是整块 numeric Core，而不是一个单独 candidate，所以它会直接进入后面的 package-level comparison。


In [7]:
#@title 第 5 步：Cheap Probe（辅助代码，可折叠）
# ============================================================
# 6. Candidate transform engine + cheap probe
# ============================================================

def _numeric_array(frame, feature):
    return pd.to_numeric(frame[feature], errors="coerce").to_numpy(dtype=float)


def _base_raw_matrix(frame, sources, fit_state=None):
    """Cheap Probe 使用的小型 raw-source matrix。"""
    cols = []
    states = []
    if fit_state is None:
        fit_state = []
        fitting = True
    else:
        fitting = False

    for i, feature in enumerate(sources):
        x = _numeric_array(frame, feature)
        if fitting:
            finite = x[np.isfinite(x)]
            median = float(np.median(finite)) if len(finite) else 0.0
            x2 = np.where(np.isfinite(x), x, median)
            mean = float(np.mean(x2))
            scale = float(np.std(x2))
            if not np.isfinite(scale) or scale < 1e-12:
                scale = 1.0
            fit_state.append({"median": median, "mean": mean, "scale": scale})
        state = fit_state[i]
        x2 = np.where(np.isfinite(x), x, state["median"])
        cols.append(((x2 - state["mean"]) / state["scale"]).reshape(-1, 1))

    X = np.hstack(cols) if cols else np.empty((len(frame), 0), dtype=float)
    return X, fit_state


def fit_candidate_state(frame, spec):
    family = spec["family"]
    sources = spec["sources"]
    params = spec["params"]
    state = {"family": family, "sources": sources, "params": params}

    if family == "winsor":
        x = _numeric_array(frame, sources[0])
        finite = x[np.isfinite(x)]
        state["lower"] = float(np.quantile(finite, params["lower_q"])) if len(finite) else 0.0
        state["upper"] = float(np.quantile(finite, params["upper_q"])) if len(finite) else 0.0

    elif family == "qbin":
        x = _numeric_array(frame, sources[0])
        finite = x[np.isfinite(x)]
        n_bins = int(params["n_bins"])
        if len(finite):
            qs = np.linspace(0, 1, n_bins + 1)[1:-1]
            edges = np.unique(np.quantile(finite, qs))
        else:
            edges = np.array([], dtype=float)
        state["edges"] = edges.tolist()

    elif family == "spline":
        x = _numeric_array(frame, sources[0])
        finite = x[np.isfinite(x)]
        median = float(np.median(finite)) if len(finite) else 0.0
        x2 = np.where(np.isfinite(x), x, median).reshape(-1, 1)
        transformer = SplineTransformer(
            n_knots=int(params["n_knots"]),
            degree=int(params["degree"]),
            knots="quantile",
            include_bias=False,
        )
        transformer.fit(x2)
        state["median"] = median
        state["transformer"] = transformer

    # For one-column continuous outputs, learn train-only normalization.
    if family in {"signed_log1p", "winsor", "ratio", "relative_change", "product"}:
        raw = transform_candidate_raw(frame, spec, state)
        finite = raw[np.isfinite(raw)]
        median = float(np.median(finite)) if finite.size else 0.0
        raw2 = np.where(np.isfinite(raw), raw, median)
        mean = float(np.mean(raw2))
        scale = float(np.std(raw2))
        if not np.isfinite(scale) or scale < 1e-12:
            scale = 1.0
        state.update({"out_median": median, "out_mean": mean, "out_scale": scale})

    return state


def transform_candidate_raw(frame, spec, state):
    family = spec["family"]
    sources = spec["sources"]

    if family in {"signed_log1p", "winsor", "qbin", "spline"}:
        x = _numeric_array(frame, sources[0])

    if family == "signed_log1p":
        return np.sign(x) * np.log1p(np.abs(x))

    if family == "winsor":
        return np.clip(x, state["lower"], state["upper"])

    if family == "ratio":
        a = _numeric_array(frame, sources[0])
        b = _numeric_array(frame, sources[1])
        eps = float(spec["params"].get("epsilon", 1e-6))
        denom = np.where(np.abs(b) < eps, np.nan, b)
        return a / denom

    if family == "relative_change":
        a = _numeric_array(frame, sources[0])
        b = _numeric_array(frame, sources[1])
        eps = float(spec["params"].get("epsilon", 1e-6))
        denom = np.maximum(np.abs(b), eps)
        return (a - b) / denom

    if family == "product":
        a = _numeric_array(frame, sources[0])
        b = _numeric_array(frame, sources[1])
        return a * b

    raise ValueError(f"No raw scalar transform for family={family}")


def transform_candidate_matrix(frame, spec, state):
    family = spec["family"]

    if family == "qbin":
        x = _numeric_array(frame, spec["sources"][0])
        edges = np.asarray(state["edges"], dtype=float)
        bins = np.digitize(x, edges, right=False)
        bins[~np.isfinite(x)] = len(edges) + 1  # explicit missing bin
        n_cols = len(edges) + 2
        rows = np.arange(len(frame))
        data = np.ones(len(frame), dtype=np.float32)
        return sparse.csr_matrix((data, (rows, bins)), shape=(len(frame), n_cols), dtype=np.float32)

    if family == "spline":
        x = _numeric_array(frame, spec["sources"][0])
        x2 = np.where(np.isfinite(x), x, state["median"]).reshape(-1, 1)
        out = state["transformer"].transform(x2).astype(np.float32)
        return sparse.csr_matrix(out)

    raw = transform_candidate_raw(frame, spec, state)
    raw = np.where(np.isfinite(raw), raw, state["out_median"])
    z = ((raw - state["out_mean"]) / state["out_scale"]).astype(np.float32)
    return sparse.csr_matrix(z.reshape(-1, 1))


def probe_one_candidate(spec, train_df, assess_df, baseline_cache=None):
    started = time.perf_counter()
    sources = spec["sources"]

    # 当前候选变换只针对 numeric source。
    if any(dtype_lookup.get(f) != "numeric" for f in sources):
        return None

    baseline_key = tuple(sources)

    if baseline_cache is not None and baseline_key in baseline_cache:
        raw_state, raw_gini = baseline_cache[baseline_key]
        X_raw_train, _ = _base_raw_matrix(train_df, sources, raw_state)
        X_raw_assess, _ = _base_raw_matrix(assess_df, sources, raw_state)
    else:
        X_raw_train, raw_state = _base_raw_matrix(train_df, sources)
        X_raw_assess, _ = _base_raw_matrix(assess_df, sources, raw_state)

        y_train_baseline = train_df[TARGET_COL].to_numpy(dtype=np.int8)
        y_assess_baseline = assess_df[TARGET_COL].to_numpy(dtype=np.int8)

        baseline = LogisticRegression(
            solver="lbfgs",
            penalty="l2",
            C=LR_SEARCH_C,
            max_iter=LR_SEARCH_MAX_ITER,
            tol=LR_SEARCH_TOL,
            random_state=RANDOM_SEED,
        )
        baseline.fit(X_raw_train, y_train_baseline)
        raw_pred = baseline.predict_proba(X_raw_assess)[:, 1]
        raw_gini = 2 * roc_auc_score(y_assess_baseline, raw_pred) - 1

        if baseline_cache is not None:
            baseline_cache[baseline_key] = (raw_state, float(raw_gini))

    state = fit_candidate_state(train_df, spec)
    X_eng_train = transform_candidate_matrix(train_df, spec, state)
    X_eng_assess = transform_candidate_matrix(assess_df, spec, state)

    X_plus_train = sparse.hstack([sparse.csr_matrix(X_raw_train), X_eng_train], format="csr")
    X_plus_assess = sparse.hstack([sparse.csr_matrix(X_raw_assess), X_eng_assess], format="csr")

    y_train = train_df[TARGET_COL].to_numpy(dtype=np.int8)
    y_assess = assess_df[TARGET_COL].to_numpy(dtype=np.int8)

    expanded = LogisticRegression(
                  solver="lbfgs",
                  penalty="l2",
                  C=LR_SEARCH_C,
                  max_iter=LR_SEARCH_MAX_ITER,
                  tol=LR_SEARCH_TOL,
                  random_state=RANDOM_SEED,
              )
    expanded.fit(X_plus_train, y_train)
    eng_pred = expanded.predict_proba(X_plus_assess)[:, 1]
    eng_gini = 2 * roc_auc_score(y_assess, eng_pred) - 1

    return {
        "candidate_id": spec["candidate_id"],
        "family": spec["family"],
        "sources_text": " | ".join(sources),
        "params_json": json.dumps(spec["params"], ensure_ascii=False, sort_keys=True),
        "raw_gini": float(raw_gini),
        "expanded_gini": float(eng_gini),
        "delta_gini": float(eng_gini - raw_gini),
        "actual_added_columns": int(X_eng_train.shape[1]),
        "fit_seconds": float(time.perf_counter() - started),
    }


PROBE_RESULTS_PATH = NB11_CHECKPOINT_DIR / f"representation_probe_results_{REPRESENTATION_EVALUATOR_VERSION}.csv"
probe_results = pd.DataFrame()

if RUN_CHEAP_PROBE:
    if not candidate_specs:
        raise RuntimeError("Candidate space 尚未生成。")

    numeric_sources = sorted({
        source
        for spec in candidate_specs
        for source in spec["sources"]
        if dtype_lookup.get(source) == "numeric"
    })

    train_probe = sample_window(
        numeric_sources,
        STAGE1_TRAIN_START,
        STAGE1_TRAIN_END,
        PROBE_TRAIN_MAX_ROWS,
        seed=RANDOM_SEED,
    )
    assess_probe = sample_window(
        numeric_sources,
        STAGE1_ASSESS_START,
        STAGE1_ASSESS_END,
        PROBE_ASSESS_MAX_ROWS,
        seed=RANDOM_SEED + 1,
    )

    if PROBE_RESULTS_PATH.exists():
        existing_probe = pd.read_csv(PROBE_RESULTS_PATH)
        current_ids = {spec["candidate_id"] for spec in candidate_specs}
        existing_probe = existing_probe.loc[
            existing_probe["candidate_id"].isin(current_ids)
        ].copy()
    else:
        existing_probe = pd.DataFrame()

    rows = (
        existing_probe.to_dict("records")
        if not existing_probe.empty
        else []
    )
    completed = (
        set(existing_probe["candidate_id"])
        if not existing_probe.empty
        else set()
    )
    baseline_cache = {}

    for i, spec in enumerate(candidate_specs, start=1):
        candidate_id = spec["candidate_id"]

        if candidate_id in completed:
            print(f"候选 {i}/{len(candidate_specs)} checkpoint 命中： {candidate_id}")
            continue

        print(f"候选 {i}/{len(candidate_specs)}： {candidate_id}")
        result = probe_one_candidate(
            spec,
            train_probe,
            assess_probe,
            baseline_cache=baseline_cache,
        )

        if result is not None:
            rows.append(result)
            completed.add(candidate_id)

            # 每完成一个 candidate 就保存一次，避免中断后全部重跑。
            (
                pd.DataFrame(rows)
                .drop_duplicates("candidate_id", keep="last")
                .to_csv(PROBE_RESULTS_PATH, index=False)
            )

    probe_results = (
        pd.DataFrame(rows)
        .drop_duplicates("candidate_id", keep="last")
        .reset_index(drop=True)
    )

elif PROBE_RESULTS_PATH.exists():
    probe_results = pd.read_csv(PROBE_RESULTS_PATH)
else:
    print("Cheap Probe 尚未运行。")

if not probe_results.empty:
    display(
        probe_results.sort_values(
            ["delta_gini", "actual_added_columns", "fit_seconds"],
            ascending=[False, True, True],
        ).head(40)
    )


候选 1/292 checkpoint 命中： signed_log1p__b6ec8c4a15eb
候选 2/292 checkpoint 命中： signed_log1p__979ee7dd2cca
候选 3/292 checkpoint 命中： signed_log1p__0d2085cdd0d0
候选 4/292 checkpoint 命中： signed_log1p__876dc59c9ec9
候选 5/292 checkpoint 命中： signed_log1p__8e03ea86f5fc
候选 6/292 checkpoint 命中： signed_log1p__65cf4b379a02
候选 7/292 checkpoint 命中： signed_log1p__62421344e730
候选 8/292 checkpoint 命中： signed_log1p__5c828bb8398f
候选 9/292 checkpoint 命中： signed_log1p__ff35236d619b
候选 10/292 checkpoint 命中： signed_log1p__1d0a7cc9e62b
候选 11/292 checkpoint 命中： signed_log1p__328b0e5b0f57
候选 12/292 checkpoint 命中： signed_log1p__dd80065f5711
候选 13/292 checkpoint 命中： signed_log1p__6c4736616fa3
候选 14/292 checkpoint 命中： signed_log1p__561487be04b0
候选 15/292 checkpoint 命中： signed_log1p__0b673c42a47b
候选 16/292 checkpoint 命中： signed_log1p__371ffb7fc2c2
候选 17/292 checkpoint 命中： signed_log1p__8245085e5be5
候选 18/292 checkpoint 命中： signed_log1p__c0010f074ff4
候选 19/292 checkpoint 命中： signed_log1p__66edbc354e67
候选 20/292 checkpoint 

,candidate_id,family,sources_text,params_json,raw_gini,expanded_gini,delta_gini,actual_added_columns,fit_seconds
75,qbin__26c0a527eaa4,qbin,price_1097A,"{""n_bins"": 8}",0.049654,0.112172,0.062518,9,0.054029
107,spline__cff5771466ad,spline,price_1097A,"{""degree"": 2, ""n_knots"": 6}",0.049654,0.106772,0.057118,6,0.092600
74,qbin__80360d2f6bb0,qbin,price_1097A,"{""n_bins"": 5}",0.049654,0.104099,0.054444,6,0.039520
2,signed_log1p__0d2085cdd0d0,signed_log1p,price_1097A,{},0.049654,0.097678,0.048024,1,0.054008
92,qbin__cade2b7cc799,qbin,periodicityofpmts_1102L_mean,"{""n_bins"": 5}",0.000556,0.037721,0.037165,3,0.056497
93,qbin__0a648c6c3dbf,qbin,periodicityofpmts_1102L_mean,"{""n_bins"": 8}",0.000556,0.037721,0.037165,3,0.057435
257,product__0c068d782bd1,product,pmtnum_254L | price_1097A,"{""tree_shared_path_count"": 94}",0.145009,0.178898,0.033889,1,0.061756
106,spline__0a12d3657c8e,spline,price_1097A,"{""degree"": 2, ""n_knots"": 4}",0.049654,0.072936,0.023281,4,0.080584
286,product__12d003badc15,product,mobilephncnt_593L | dpdmaxdateyear_596T_mean,"{""tree_shared_path_count"": 63}",0.273727,0.295625,0.021898,1,0.057983
237,ratio__2630632db6b2,ratio,overdueamountmaxdatemonth_494T_max | overdueam...,"{""epsilon"": 1e-06}",0.005459,0.023520,0.018062,1,0.059544


## 9. Family-wise Candidate Screening

Cheap Probe 只需要回答一个简单问题：

> **这个 transformation 有没有明显的增量迹象？**

因此这里不再建立多目标 Pareto frontier。

每个 family 内：

```text
1. 先要求 ΔGini >= PROBE_MIN_DELTA_GINI
2. 按 ΔGini 从高到低排序
3. added columns 只作为 tie-break
4. 每个 family 最多保留固定数量
```

这样既不会让候选数量无限扩大，
也不会为了 screening 阶段加入过多工程化判断。

Cheap Probe 的结果仍然只是下一阶段的候选来源，
不是最终模型结论。


In [8]:
#@title 第 6 步：每个 family 保留少量 survivors
# ============================================================
# Simple family-wise probe survivors
# ============================================================

def choose_probe_survivors(results):
    survivors = []

    for family, group in results.groupby("family"):
        group = group.loc[
            group["delta_gini"].ge(PROBE_MIN_DELTA_GINI)
        ].copy()

        if group.empty:
            continue

        group = (
            group
            .sort_values(
                ["delta_gini", "actual_added_columns"],
                ascending=[False, True],
            )
            .drop_duplicates("candidate_id")
            .head(MAX_PROBE_SURVIVORS_PER_FAMILY)
        )

        survivors.append(group)

    return (
        pd.concat(survivors, ignore_index=True)
        if survivors
        else pd.DataFrame()
    )


probe_survivors = (
    choose_probe_survivors(probe_results)
    if not probe_results.empty
    else pd.DataFrame()
)

if not probe_survivors.empty:
    probe_survivors.to_csv(
        NB11_OUTPUT_DIR / "representation_probe_survivors.csv",
        index=False,
    )

    display(
        probe_survivors
        .groupby("family")
        .agg(
            survivors=("candidate_id", "count"),
            mean_delta_gini=("delta_gini", "mean"),
            columns_added=("actual_added_columns", "sum"),
        )
        .reset_index()
    )


,family,survivors,mean_delta_gini,columns_added
0,product,15,0.007395,15
1,qbin,15,0.016265,79
2,ratio,15,0.005525,15
3,relative_change,15,0.002490,15
4,signed_log1p,15,0.004911,15
5,spline,15,0.007109,70
6,winsor,15,0.000155,15


## 10. Stage 2 — Pair Feature Candidate Space

Pair feature 至少有两种不同来源。

### A. Arithmetic pair

例如：

```text
payment / credit_amount
recent_mean / long_term_mean
(current_balance - historical_balance) / |historical_balance|
```

它强调的是 **业务量纲 / 相对关系**。

因此本册只在名称上属于同一语义 stem 的 numeric variables 之间提议：

```text
ratio
relative_change
```

### B. Interaction pair

例如：

```text
A × B
```

它不一定有直接业务量纲含义，而是在问：

> A 的影响是否取决于 B？

因此本册不做 all-pairs product，而只使用 NB8 已经保存的：

```text
Tree shared-path pair evidence
```

作为 proposal mechanism。

Friedman & Popescu 的 RuleFit 正是类似思想：
Tree 可以提出 rules / interactions，随后由线性模型进行选择。

Bien, Taylor & Tibshirani (2013) 的 hierarchical interaction 研究则说明：
用 hierarchy 约束 interaction space 是一种有效的结构控制方法。

但这里不会把 strong hierarchy 当成绝对真理。

本册采用的是 **weak computational hierarchy**：

```text
Core × Core
→ 正常允许

Core × Rescue
→ 有 pair evidence 时允许

Rescue × Rescue
→ 只有很强的 pair evidence 才值得付预算
```

<details>
<summary><strong>📘 最小例子：为什么 ratio 和 product 不应该混在一起？</strong></summary>

```text
income / credit_amount
```

有一个直接解释：

```text
还款能力相对于借款规模
```

但：

```text
income × credit_amount
```

通常没有同样直接的单位解释。

它更像：

```text
一个 interaction basis
```

所以 ratio / relative change 可以由业务语义提出；
product 更适合由 interaction evidence 提出。

至于 `A-B`：如果 `A`、`B` 本身都已经作为 raw main effects 留在 LR 里，它通常只是已有列的线性组合，所以本册不单独为它付搜索预算。

</details>


## 11. Stage 3 — Representation Package Construction

Cheap Probe 回答的是：

> **一个 transformation 单独有没有继续搜索价值？**

但最终 LR 需要同时使用一组 representation，所以这里从：

```text
Candidate Search
```

转到：

```text
Package Search
```

Local FE 不搜索所有 survivor subsets，而只组成少量 nested packages，例如：

```text
Raw Core
Raw Core + Top 10 Unary
Raw Core + Top 20 Unary

再在这些 unary package 上
加入少量 Top Pair candidates
```

这继续沿用 NB10 的原则：

> **不搜索任意 subset，而只测试少量有方向的 package。**

### PCA 作为 Package-Level Challenger

PCA 是联合学习出来的 latent basis，不适合拆成单个 component 做 Cheap Probe。

因此 Stage 3 直接比较：

```text
raw_core
Local FE packages
pca_numeric50
pca_numeric100
```

其中 `pca_numericK` 的结构是：

```text
numeric Core
        ↓ PCA(K)

missing indicators
categorical One-Hot
        ↓
与 PCA components 拼接
```

这样 Stage 3 真正回答的是：

> **Raw Core、少量 Local FE、还是 numeric PCA，更值得继续做 temporal robustness check？**


In [9]:
#@title 第 7 步：组成 Representation Packages（辅助代码，可折叠）
# ============================================================
# 8. Build Local-FE packages + Global-DR challenger packages
# ============================================================

spec_lookup = {
    spec["candidate_id"]: spec
    for spec in candidate_specs
}


def sorted_survivor_ids(families):
    if probe_survivors.empty:
        return []

    part = (
        probe_survivors
        .loc[
            probe_survivors[
                "family"
            ].isin(
                families
            )
        ]
        .copy()
    )

    return (
        part
        .sort_values(
            [
                "delta_gini",
                "actual_added_columns",
            ],
            ascending=[
                False,
                True,
            ],
        )[
            "candidate_id"
        ]
        .drop_duplicates()
        .tolist()
    )


unary_ids = sorted_survivor_ids(
    {
        "signed_log1p",
        "winsor",
        "qbin",
        "spline",
    }
)

pair_ids = sorted_survivor_ids(
    {
        "ratio",
        "relative_change",
        "product",
    }
)


# ------------------------------------------------------------
# A. Local / interpretable FE packages
# ------------------------------------------------------------
representation_packages = {
    "raw_core": [],
}

unary_backbones = {
    "raw_core": [],
}

for k in UNARY_PACKAGE_SIZES:

    actual = min(
        k,
        len(unary_ids),
    )

    if actual > 0:

        name = (
            f"unary_top{actual}"
        )

        ids = (
            unary_ids[:actual]
        )

        representation_packages[
            name
        ] = ids

        unary_backbones[
            name
        ] = ids


# Pair value can depend on how much unary representation
# is already present.  Test a small nested grid instead of
# assuming the largest unary package is automatically best.
for (
    unary_name,
    base_ids,
) in list(
    unary_backbones.items()
):

    if unary_name == "raw_core":
        continue

    for k in PAIR_PACKAGE_SIZES:

        actual = min(
            k,
            len(pair_ids),
        )

        if actual > 0:

            representation_packages[
                f"{unary_name}"
                f"__pair_top{actual}"
            ] = [
                *base_ids,
                *pair_ids[:actual],
            ]


# ------------------------------------------------------------
# B. Unified package definitions
# ------------------------------------------------------------
#
# Local FE:
#   candidate_ids point to candidate_specs.
#
# Global DR:
#   operates on the whole NB10 Core block and therefore does
#   not pretend to be a single-candidate transform.
package_definitions = {}

for (
    name,
    candidate_ids,
) in representation_packages.items():

    package_definitions[
        name
    ] = {
        "mode": (
            "local_fe"
        ),
        "candidate_ids": list(
            candidate_ids
        ),
    }


if ENABLE_GLOBAL_DR_BASELINES:

    # Global challenger 只保留 PCA。
    for k in DR_COMPONENT_OPTIONS:
        package_definitions[f"pca_numeric{k}"] = {
            "mode": "pca_numeric",
            "n_components": int(k),
            "candidate_ids": [],
            "source_scope": "lr_core_features",
        }


# ------------------------------------------------------------
# Identify one readable Local-FE reference baseline.
# It is NOT a duplicated package; it points to an existing
# nested Local-FE package so no extra training is created.
# ------------------------------------------------------------
preferred_local_references = [
    "unary_top20__pair_top10",
    "unary_top20",
    "unary_top10__pair_top10",
    "unary_top10",
    "unary_top30__pair_top10",
    "unary_top30",
]

LOCAL_FE_REFERENCE_PACKAGE = next(
    (
        name
        for name
        in preferred_local_references
        if name
        in package_definitions
    ),
    "raw_core",
)


# ------------------------------------------------------------
# Package catalog
# ------------------------------------------------------------
package_catalog_rows = []

for (
    name,
    package_def,
) in package_definitions.items():

    mode = package_def[
        "mode"
    ]

    candidate_ids = (
        package_def.get(
            "candidate_ids",
            [],
        )
    )

    if mode == "local_fe":

        specs = [
            spec_lookup[cid]
            for cid
            in candidate_ids
        ]

        required_sources = sorted({
            source
            for spec
            in specs
            for source
            in spec["sources"]
        })

        # Weak hierarchy / interpretability support:
        # raw source terms remain available whenever
        # a local transform is used.
        raw_features = sorted(
            set(
                lr_core_features
            )
            | set(
                required_sources
            )
        )

        estimated_added_columns = int(
            sum(
                estimate_added_columns(
                    spec
                )
                for spec
                in specs
            )
        )

        requested_components = np.nan

    else:

        specs = []

        raw_features = list(
            lr_core_features
        )

        estimated_added_columns = np.nan

        requested_components = int(
            package_def[
                "n_components"
            ]
        )


    package_catalog_rows.append({
        "package": name,
        "package_mode": mode,
        "engineered_candidate_count": len(
            candidate_ids
        ),
        "raw_source_count": len(
            raw_features
        ),
        "estimated_added_columns": (
            estimated_added_columns
        ),
        "requested_components": (
            requested_components
        ),
        "candidate_ids": (
            candidate_ids
        ),
        "raw_features": (
            raw_features
        ),
    })


package_catalog = pd.DataFrame(
    package_catalog_rows
)


BASELINE_PACKAGE_NAMES = [
    name
    for name
    in [
        "raw_core",
        LOCAL_FE_REFERENCE_PACKAGE,
        "pca_numeric50",
        "pca_numeric100",
    ]
    if name
    in package_definitions
]

# De-duplicate when Local-FE reference falls back to raw_core.
BASELINE_PACKAGE_NAMES = list(
    dict.fromkeys(
        BASELINE_PACKAGE_NAMES
    )
)


print(
    "Local FE 参考方案：",
    LOCAL_FE_REFERENCE_PACKAGE,
)

print(
    "基线 / 对照方案：",
    BASELINE_PACKAGE_NAMES,
)


if not package_catalog.empty:

    display(
        package_catalog[[
            "package",
            "package_mode",
            "engineered_candidate_count",
            "raw_source_count",
            "estimated_added_columns",
            "requested_components",
        ]]
    )


Local FE 参考方案： unary_top20__pair_top10
基线 / 对照方案： ['raw_core', 'unary_top20__pair_top10', 'pca_numeric50', 'pca_numeric100']


,package,package_mode,engineered_candidate_count,raw_source_count,estimated_added_columns,requested_components
0,raw_core,local_fe,0,100,0.0,NaN
1,unary_top10,local_fe,10,104,53.0,NaN
2,unary_top20,local_fe,20,109,109.0,NaN
3,unary_top30,local_fe,30,114,155.0,NaN
4,unary_top40,local_fe,40,118,175.0,NaN
5,unary_top10__pair_top10,local_fe,20,110,63.0,NaN
6,unary_top10__pair_top20,local_fe,30,114,73.0,NaN
7,unary_top20__pair_top10,local_fe,30,115,119.0,NaN
8,unary_top20__pair_top20,local_fe,40,119,129.0,NaN
9,unary_top30__pair_top10,local_fe,40,120,165.0,NaN


## 12. Full-Core Package Evaluation

Cheap Probe 有一个故意接受的盲点：

```text
它只比较局部 source representation
```

但一个 feature 的增量价值依赖模型 context。

例如：

```text
ratio(A,B)
```

单独看可能很强；但如果 Core 已经包含高度相关的 `C`，它进入完整模型以后可能没有新增价值。

反过来，一个单变量 probe 只表现普通的 spline，在控制 Core 其他变量后也可能提供独立信息。

所以：

```text
Cheap Probe
→ 负责大量淘汰

Package Evaluation
→ 回到完整 Core context 中重新判断
```

这相当于：

```text
marginal screening
→ conditional / model-context evaluation
```

### PCA 为什么也必须在完整 Package 层比较？

PCA 改变的是整块坐标系：

```text
X1, X2, ..., Xp
        ↓
PC1, PC2, ..., PCK
```

它没有自然的“单变量 baseline”，所以只能作为完整 package 与 Raw Core / Local FE 使用同一个 evaluator 比较。

比较时保持：

```text
同一个 LR learner
同一个 Train / Assess window
同一个 weekly stability evaluator
同一个 F1–F4 repeatability test
```

只有 representation 本身不同。

`explained_variance_ratio` 会被记录，但只用于描述 PCA 压缩了多少输入变化，不直接决定它是否晋级；真正的 selection evidence 仍然来自 Gini、时间稳定性和 representation dimension。


In [10]:
#@title 第 8 步：完整 Package Evaluator（辅助代码，可折叠）
# ============================================================
# Frozen NB7 preprocessing + Local FE + PCA
# ============================================================

LR_MISSING_TOKEN = "__HC_MISSING__"
LR_RARE_TOKEN = "__HC_RARE__"

nb7_dtype = nb7_catalog.set_index("feature")["dtype_family"].to_dict()
numeric_stats_indexed = numeric_stats.set_index("feature")
LR_STATE_CACHE = {}


def make_lr_state(features):
    """读取 Notebook 7 已冻结的 LR preprocessing rule。"""
    cache_key = tuple(features)
    if cache_key in LR_STATE_CACHE:
        return LR_STATE_CACHE[cache_key]

    numeric = [f for f in features if nb7_dtype.get(f) == "numeric"]
    categorical = [
        f for f in features
        if nb7_dtype.get(f) in {"categorical", "boolean"}
    ]

    stats = numeric_stats_indexed.reindex(numeric)
    required_cols = ["median", "mean_after_imputation", "scale_after_imputation"]
    if numeric and stats[required_cols].isna().any().any():
        missing = stats.index[stats[required_cols].isna().any(axis=1)].tolist()
        raise ValueError(f"NB7 数值预处理规则缺失：{missing[:10]}")

    medians = stats["median"].to_numpy(dtype=np.float32) if numeric else np.array([], dtype=np.float32)
    means = stats["mean_after_imputation"].to_numpy(dtype=np.float32) if numeric else np.array([], dtype=np.float32)
    scales = stats["scale_after_imputation"].to_numpy(dtype=np.float32) if numeric else np.array([], dtype=np.float32)
    scales[~np.isfinite(scales) | (np.abs(scales) < 1e-12)] = 1.0

    missing_features = (
        stats.index[stats["has_missing"].fillna(False).astype(bool)].tolist()
        if numeric and "has_missing" in stats.columns
        else []
    )
    missing_indices = [numeric.index(f) for f in missing_features]

    category_lists = []
    kept_sets = {}
    for feature in categorical:
        if feature not in lr_category_lists:
            raise KeyError(f"NB7 categorical rule 缺少 feature：{feature}")
        cats = [str(x) for x in lr_category_lists[feature]]
        if not cats:
            raise ValueError(f"NB7 categorical rule 为空：{feature}")
        category_lists.append(cats)
        kept_sets[feature] = set(cats)

    if categorical:
        kwargs = dict(categories=category_lists, handle_unknown="ignore", dtype=np.float32)
        try:
            encoder = OneHotEncoder(sparse_output=True, **kwargs)
        except TypeError:
            encoder = OneHotEncoder(sparse=True, **kwargs)
        dummy = pd.DataFrame({
            feature: [category_lists[i][0]]
            for i, feature in enumerate(categorical)
        })
        encoder.fit(dummy)
    else:
        encoder = None

    state = {
        "features": list(features),
        "numeric": numeric,
        "categorical": categorical,
        "medians": medians,
        "means": means,
        "scales": scales,
        "missing_indices": missing_indices,
        "kept_sets": kept_sets,
        "encoder": encoder,
    }
    LR_STATE_CACHE[cache_key] = state
    return state


def transform_lr_blocks(frame, state):
    """返回 numeric / missing-indicator / categorical 三个 block。"""
    n = len(frame)
    numeric = state["numeric"]

    if numeric:
        X_num = frame[numeric].to_numpy(dtype=np.float32, na_value=np.nan)
        missing_mask = ~np.isfinite(X_num)
        if missing_mask.any():
            rows, cols = np.where(missing_mask)
            X_num[rows, cols] = state["medians"][cols]

        # 与 NB7 保持一致：median fill → 减 mean → 除 scale。
        X_num -= state["means"]
        X_num /= state["scales"]

        if state["missing_indices"]:
            X_miss = missing_mask[:, state["missing_indices"]].astype(np.float32)
            miss_sparse = sparse.csr_matrix(X_miss, dtype=np.float32)
        else:
            miss_sparse = sparse.csr_matrix((n, 0), dtype=np.float32)
    else:
        X_num = np.empty((n, 0), dtype=np.float32)
        miss_sparse = sparse.csr_matrix((n, 0), dtype=np.float32)

    categorical = state["categorical"]
    if categorical:
        mapped = {}
        for feature in categorical:
            series = frame[feature].astype("string").fillna(LR_MISSING_TOKEN)
            kept = state["kept_sets"][feature]
            series = series.where(series.isin(kept), LR_RARE_TOKEN)
            mapped[feature] = series.astype(object)
        cat_sparse = state["encoder"].transform(pd.DataFrame(mapped, index=frame.index))
    else:
        cat_sparse = sparse.csr_matrix((n, 0), dtype=np.float32)

    return X_num, miss_sparse, cat_sparse


def transform_lr_raw(frame, state):
    X_num, X_miss, X_cat = transform_lr_blocks(frame, state)
    return sparse.hstack(
        [sparse.csr_matrix(X_num, dtype=np.float32), X_miss, X_cat],
        format="csr",
        dtype=np.float32,
    )


def raw_design_width(state):
    categorical_columns = (
        int(sum(len(cats) for cats in state["encoder"].categories_))
        if state["encoder"] is not None
        else 0
    )
    return int(
        len(state["numeric"])
        + len(state["missing_indices"])
        + categorical_columns
    )


def _to_numpy(value, dtype=None):
    if hasattr(value, "get"):
        value = value.get()
    elif hasattr(value, "to_numpy") and not isinstance(value, np.ndarray):
        value = value.to_numpy()
    return np.asarray(value, dtype=dtype)


def _new_lr():
    return LogisticRegression(
        solver="lbfgs",
        penalty="l2",
        C=LR_SEARCH_C,
        max_iter=LR_SEARCH_MAX_ITER,
        tol=LR_SEARCH_TOL,
        random_state=RANDOM_SEED,
    )


# ------------------------------------------------------------
# Core design cache
# ------------------------------------------------------------
# NB10 Core 在所有 package 中都会反复使用。
# 这里按 parquet row group 只做一次 NB7 preprocessing，后续 package 直接复用。
CORE_STATE = make_lr_state(list(lr_core_features))
CORE_NUMERIC_COLUMNS = len(CORE_STATE["numeric"])
CORE_DESIGN_COLUMNS = raw_design_width(CORE_STATE)
CORE_SIDE_COLUMNS = CORE_DESIGN_COLUMNS - CORE_NUMERIC_COLUMNS

_core_cache_id = hashlib.sha256(
    json.dumps({
        "core_features": list(lr_core_features),
        "nb7": NB7_PREPROCESSING_SIGNATURE,
        "nb10": NB10_SOURCE_SIGNATURE,
        "version": REPRESENTATION_EVALUATOR_VERSION,
    }, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

CORE_MATRIX_CACHE_DIR = LOCAL_TRAIN_CACHE_DIR / f"core_design_{_core_cache_id}"
CORE_MATRIX_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_CORE_ROW_GROUP_MEMORY = {}


def get_core_row_group_matrix(rg):
    rg = int(rg)
    # 只在内存里保留最近读取的一块，避免无界增长。
    if rg in _CORE_ROW_GROUP_MEMORY:
        return _CORE_ROW_GROUP_MEMORY[rg]

    path = CORE_MATRIX_CACHE_DIR / f"rg_{rg:04d}.npz"
    if path.exists():
        X = sparse.load_npz(path).tocsr().astype(np.float32, copy=False)
    else:
        table = train_parquet.read_row_group(
            rg,
            columns=list(lr_core_features),
            use_threads=True,
        )
        frame = table.to_pandas().reset_index(drop=True)
        X = transform_lr_raw(frame, CORE_STATE)
        temp_path = path.with_suffix(".tmp.npz")
        sparse.save_npz(temp_path, X, compressed=False)
        temp_path.replace(path)
        del table, frame

    _CORE_ROW_GROUP_MEMORY.clear()
    _CORE_ROW_GROUP_MEMORY[rg] = X
    return X



# ------------------------------------------------------------
# Local FE
# ------------------------------------------------------------
LOCAL_FE_FIT_SAMPLE_ACTIVE = {"key": None, "frame": None}


def _local_fe_fit_source_pool():
    ids = {
        cid
        for package_def in package_definitions.values()
        if package_def.get("mode") == "local_fe"
        for cid in package_def.get("candidate_ids", [])
    }
    return sorted({
        source
        for cid in ids
        for source in spec_lookup[cid]["sources"]
    })


def get_local_fe_fit_sample(train_start, train_end):
    key = (int(train_start), int(train_end))
    if LOCAL_FE_FIT_SAMPLE_ACTIVE["key"] != key:
        source_pool = _local_fe_fit_source_pool()
        LOCAL_FE_FIT_SAMPLE_ACTIVE["frame"] = sample_window(
            source_pool,
            train_start,
            train_end,
            PROFILE_SAMPLE_MAX_ROWS,
            seed=RANDOM_SEED + int(train_end),
        )
        LOCAL_FE_FIT_SAMPLE_ACTIVE["key"] = key
    return LOCAL_FE_FIT_SAMPLE_ACTIVE["frame"]


def fit_engineered_states(specs, train_start, train_end):
    if not specs:
        return {}
    fit_sample = get_local_fe_fit_sample(train_start, train_end)
    return {
        spec["candidate_id"]: fit_candidate_state(fit_sample, spec)
        for spec in specs
    }


def transform_engineered(frame, specs, states):
    if not specs:
        return sparse.csr_matrix((len(frame), 0), dtype=np.float32)
    blocks = [
        transform_candidate_matrix(frame, spec, states[spec["candidate_id"]])
        for spec in specs
    ]
    return sparse.hstack(blocks, format="csr", dtype=np.float32)


def engineered_design_width_from_states(specs, states):
    total = 0
    for spec in specs:
        family = spec["family"]
        state = states[spec["candidate_id"]]
        if family == "qbin":
            total += len(state["edges"]) + 2
        elif family == "spline":
            total += int(state["transformer"].n_features_out_)
        else:
            total += 1
    return int(total)


def _collect_local_design(specs, eng_states, week_start, week_end):
    required_sources = sorted({
        source for spec in specs for source in spec["sources"]
    })
    extra_features = sorted(set(required_sources) - set(lr_core_features))
    extra_state = make_lr_state(extra_features) if extra_features else None

    core_parts = []
    side_parts = []

    for rg, local_indices in get_window_row_groups(week_start, week_end):
        X_core = get_core_row_group_matrix(rg)[local_indices]
        core_parts.append(X_core)

        if required_sources:
            table = train_parquet.read_row_group(
                int(rg),
                columns=required_sources,
                use_threads=True,
            )
            if local_indices.size != table.num_rows:
                table = table.take(pa.array(local_indices))
            frame = table.to_pandas().reset_index(drop=True)

            side_blocks = []
            if extra_features:
                side_blocks.append(transform_lr_raw(frame, extra_state))
            if specs:
                side_blocks.append(transform_engineered(frame, specs, eng_states))

            X_side = (
                sparse.hstack(side_blocks, format="csr", dtype=np.float32)
                if side_blocks
                else sparse.csr_matrix((len(frame), 0), dtype=np.float32)
            )
            side_parts.append(X_side)
            del table, frame

    X_core_all = sparse.vstack(core_parts, format="csr", dtype=np.float32)
    if side_parts:
        X_side_all = sparse.vstack(side_parts, format="csr", dtype=np.float32)
        X = sparse.hstack([X_core_all, X_side_all], format="csr", dtype=np.float32)
    else:
        X = X_core_all

    return X, extra_state


def fit_local_representation(specs, train_start, train_end):
    eng_states = fit_engineered_states(specs, train_start, train_end)
    X_train, extra_state = _collect_local_design(
        specs, eng_states, train_start, train_end
    )
    y_train = get_window_meta(train_start, train_end)[TARGET_COL].to_numpy(dtype=np.int8)
    model = _new_lr()
    model.fit(X_train, y_train)
    del X_train, y_train
    gc.collect()
    return model, eng_states, extra_state


def predict_local_representation(model, specs, eng_states, assess_start, assess_end):
    X_assess, _ = _collect_local_design(
        specs, eng_states, assess_start, assess_end
    )
    pred = _to_numpy(model.predict_proba(X_assess)[:, 1], dtype=np.float32).reshape(-1)
    meta = get_window_meta(assess_start, assess_end).copy()
    if len(meta) != len(pred):
        raise ValueError(f"预测对齐失败：meta={len(meta)}, pred={len(pred)}")
    meta["prediction"] = pred
    del X_assess, pred
    gc.collect()
    return meta


# ------------------------------------------------------------
# PCA representation
# ------------------------------------------------------------
def get_pca_fit_numeric_sample(train_start, train_end):
    selector = _build_sample_selector(
        train_start,
        train_end,
        DR_FIT_SAMPLE_MAX_ROWS,
        seed=RANDOM_SEED + 10_000 + int(train_end),
    )
    if selector.empty:
        raise RuntimeError("PCA fit sample 为空。")

    parts = []
    observed_weeks = []
    for rg, part in selector.groupby("row_group", sort=True):
        local_indices = part["local_index"].to_numpy(dtype=np.int32)
        X_rg = get_core_row_group_matrix(int(rg))[local_indices]
        parts.append(X_rg[:, :CORE_NUMERIC_COLUMNS].toarray().astype(np.float32))
        observed_weeks.append(ROW_GROUP_WEEK_CACHE[int(rg)][local_indices])

    X_num = np.vstack(parts)
    weeks = np.concatenate(observed_weeks)
    return X_num, weeks


def _safe_component_count(requested, n_rows, n_columns):
    if n_rows <= 1 or n_columns <= 1:
        raise ValueError("PCA 输入维度不足。")
    return max(1, min(int(requested), int(n_rows) - 1, int(n_columns)))


def fit_pca_state(package_def, train_start, train_end):
    if CORE_NUMERIC_COLUMNS == 0:
        raise ValueError("PCA baseline 至少需要一个 numeric Core feature。")

    X_fit, observed_weeks = get_pca_fit_numeric_sample(train_start, train_end)
    actual_components = _safe_component_count(
        package_def["n_components"], X_fit.shape[0], X_fit.shape[1]
    )
    reducer = PCA(n_components=actual_components, svd_solver="auto")
    reducer.fit(X_fit)

    explained = float(np.sum(_to_numpy(reducer.explained_variance_ratio_, dtype=float)))
    observed_min = int(np.min(observed_weeks))
    observed_max = int(np.max(observed_weeks))
    if observed_min < int(train_start) or observed_max > int(train_end):
        raise AssertionError("PCA fit sample 越过当前 training window。")

    return {
        "reducer": reducer,
        "actual_components": int(actual_components),
        "design_columns": int(actual_components + CORE_SIDE_COLUMNS),
        "explained_variance_ratio": explained,
        "reducer_fit_rows": int(X_fit.shape[0]),
        "reducer_fit_week_start": int(train_start),
        "reducer_fit_week_end": int(train_end),
        "reducer_fit_observed_week_min": observed_min,
        "reducer_fit_observed_week_max": observed_max,
    }


def collect_pca_design(pca_state, week_start, week_end):
    reducer = pca_state["reducer"]
    parts = []
    for rg, local_indices in get_window_row_groups(week_start, week_end):
        X_core = get_core_row_group_matrix(rg)[local_indices]
        X_num = X_core[:, :CORE_NUMERIC_COLUMNS].toarray().astype(np.float32)
        X_pc = _to_numpy(reducer.transform(X_num), dtype=np.float32)
        X_side = X_core[:, CORE_NUMERIC_COLUMNS:]
        parts.append(
            sparse.hstack(
                [sparse.csr_matrix(X_pc, dtype=np.float32), X_side],
                format="csr",
                dtype=np.float32,
            )
        )
    if not parts:
        raise RuntimeError(f"Week {week_start}–{week_end} 没有可用数据。")
    return sparse.vstack(parts, format="csr", dtype=np.float32)


def fit_pca_representation(package_def, train_start, train_end):
    pca_state = fit_pca_state(package_def, train_start, train_end)
    X_train = collect_pca_design(pca_state, train_start, train_end)
    y_train = get_window_meta(train_start, train_end)[TARGET_COL].to_numpy(dtype=np.int8)
    model = _new_lr()
    model.fit(X_train, y_train)
    del X_train, y_train
    gc.collect()
    return model, pca_state


def predict_pca_representation(model, pca_state, assess_start, assess_end):
    X_assess = collect_pca_design(pca_state, assess_start, assess_end)
    pred = _to_numpy(model.predict_proba(X_assess)[:, 1], dtype=np.float32).reshape(-1)
    meta = get_window_meta(assess_start, assess_end).copy()
    if len(meta) != len(pred):
        raise ValueError(f"预测对齐失败：meta={len(meta)}, pred={len(pred)}")
    meta["prediction"] = pred
    del X_assess, pred
    gc.collect()
    return meta


In [11]:
#@title 第 9 步：时间指标与 Package Checkpoint（辅助代码，可折叠）

def calculate_weekly_gini(prediction_frame):
    rows = []
    for week, group in prediction_frame.groupby(WEEK_COL, sort=True):
        if group[TARGET_COL].nunique() < 2:
            gini = np.nan
        else:
            auc = roc_auc_score(group[TARGET_COL], group["prediction"])
            gini = 2.0 * auc - 1.0
        rows.append({
            WEEK_COL: int(week),
            "gini": gini,
            "sample_count": len(group),
        })
    return pd.DataFrame(rows)


def temporal_metrics(weekly_gini):
    table = weekly_gini.dropna(subset=["gini"]).sort_values(WEEK_COL)
    if len(table) < 3:
        return {
            "mean_weekly_gini": np.nan,
            "weekly_gini_slope": np.nan,
            "negative_slope": np.nan,
            "weekly_residual_sd": np.nan,
            "stability_metric": np.nan,
        }

    x = table[WEEK_COL].to_numpy(dtype=float)
    y = table["gini"].to_numpy(dtype=float)
    slope, intercept = np.polyfit(x, y, deg=1)
    residual_sd = float(np.std(y - (slope * x + intercept), ddof=0))
    mean_gini = float(np.mean(y))

    return {
        "mean_weekly_gini": mean_gini,
        "weekly_gini_slope": float(slope),
        "negative_slope": max(0.0, -float(slope)),
        "weekly_residual_sd": residual_sd,
        "stability_metric": (
            mean_gini
            + 88.0 * min(0.0, float(slope))
            - 0.5 * residual_sd
        ),
    }


def package_fingerprint(name, package_def, train_start, train_end, assess_start, assess_end):
    payload = json.dumps(
        {
            "evaluator_version": REPRESENTATION_EVALUATOR_VERSION,
            "package": name,
            "package_definition": package_def,
            "train": [int(train_start), int(train_end)],
            "assess": [int(assess_start), int(assess_end)],
            "lr_C": LR_SEARCH_C,
            "lr_max_iter": LR_SEARCH_MAX_ITER,
            "dr_fit_sample_max_rows": DR_FIT_SAMPLE_MAX_ROWS,
            "preprocessing_policy": PREPROCESSING_POLICY,
            "nb7_preprocessing_signature": NB7_PREPROCESSING_SIGNATURE,
            "nb10_source_signature": NB10_SOURCE_SIGNATURE,
            "local_candidate_specs": [
                {
                    "candidate_id": cid,
                    "family": spec_lookup[cid]["family"],
                    "sources": list(spec_lookup[cid]["sources"]),
                    "params": dict(spec_lookup[cid]["params"]),
                }
                for cid in package_def.get("candidate_ids", [])
            ],
        },
        sort_keys=True,
        ensure_ascii=False,
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:20]


def evaluate_package(name, package_def, train_start, train_end, assess_start, assess_end):
    assert_temporal_split(
        train_start, train_end, assess_start, assess_end,
        label=f"evaluate_package[{name}]",
    )

    key = package_fingerprint(
        name, package_def, train_start, train_end, assess_start, assess_end
    )
    summary_path = NB11_CHECKPOINT_DIR / f"package__{key}__summary.json"
    weekly_path = NB11_CHECKPOINT_DIR / f"package__{key}__weekly.csv"

    if summary_path.exists() and weekly_path.exists():
        print(f"checkpoint 命中：{name} | {train_start}-{train_end} → {assess_start}-{assess_end}")
        with open(summary_path, encoding="utf-8") as file:
            summary = json.load(file)
        return summary, pd.read_csv(weekly_path)

    started = time.perf_counter()
    mode = package_def["mode"]

    reducer_fit_rows = None
    reducer_fit_week_start = None
    reducer_fit_week_end = None
    reducer_fit_observed_week_min = None
    reducer_fit_observed_week_max = None
    reducer_fit_policy = "not_applicable"
    explained_variance_ratio = None
    representation_components = 0

    if mode == "local_fe":
        candidate_ids = list(package_def.get("candidate_ids", []))
        specs = [spec_lookup[cid] for cid in candidate_ids]
        required_sources = sorted({s for spec in specs for s in spec["sources"]})
        extra_features = sorted(set(required_sources) - set(lr_core_features))

        model, eng_states, extra_state = fit_local_representation(
            specs, train_start, train_end
        )
        pred_frame = predict_local_representation(
            model, specs, eng_states, assess_start, assess_end
        )

        actual_eng_cols = engineered_design_width_from_states(specs, eng_states)
        extra_width = raw_design_width(extra_state) if extra_state is not None else 0
        design_columns = int(CORE_DESIGN_COLUMNS + extra_width + actual_eng_cols)
        raw_source_count = len(set(lr_core_features) | set(extra_features))
        del model, eng_states, extra_state

    elif mode == "pca_numeric":
        candidate_ids = []
        actual_eng_cols = 0
        raw_source_count = len(lr_core_features)

        model, pca_state = fit_pca_representation(package_def, train_start, train_end)
        pred_frame = predict_pca_representation(
            model, pca_state, assess_start, assess_end
        )

        representation_components = int(pca_state["actual_components"])
        design_columns = int(pca_state["design_columns"])
        explained_variance_ratio = float(pca_state["explained_variance_ratio"])
        reducer_fit_rows = int(pca_state["reducer_fit_rows"])
        reducer_fit_week_start = int(pca_state["reducer_fit_week_start"])
        reducer_fit_week_end = int(pca_state["reducer_fit_week_end"])
        reducer_fit_observed_week_min = int(pca_state["reducer_fit_observed_week_min"])
        reducer_fit_observed_week_max = int(pca_state["reducer_fit_observed_week_max"])
        reducer_fit_policy = REDUCER_FIT_POLICY
        del model, pca_state

    else:
        raise ValueError(f"未知 representation mode：{mode}")

    auc = roc_auc_score(pred_frame[TARGET_COL], pred_frame["prediction"])
    weekly = calculate_weekly_gini(pred_frame)
    tm = temporal_metrics(weekly)

    summary = {
        "package": name,
        "package_mode": mode,
        "evaluation_role": EVALUATION_ROLE,
        "train_week_start": int(train_start),
        "train_week_end": int(train_end),
        "assess_week_start": int(assess_start),
        "assess_week_end": int(assess_end),
        "preprocessing_policy": PREPROCESSING_POLICY,
        "preprocessing_fit_week_start": PREPROCESSING_FIT_WEEK_START,
        "preprocessing_fit_week_end": PREPROCESSING_FIT_WEEK_END,
        "reducer_fit_policy": reducer_fit_policy,
        "reducer_fit_rows": reducer_fit_rows,
        "reducer_fit_week_start": reducer_fit_week_start,
        "reducer_fit_week_end": reducer_fit_week_end,
        "reducer_fit_observed_week_min": reducer_fit_observed_week_min,
        "reducer_fit_observed_week_max": reducer_fit_observed_week_max,
        "nb7_preprocessing_signature": NB7_PREPROCESSING_SIGNATURE,
        "nb10_source_signature": NB10_SOURCE_SIGNATURE,
        "candidate_count": len(candidate_ids),
        "raw_source_count": int(raw_source_count),
        "engineered_columns": int(actual_eng_cols),
        "representation_components": int(representation_components),
        "design_columns": int(design_columns),
        "explained_variance_ratio": explained_variance_ratio,
        "auc": float(auc),
        "gini": float(2.0 * auc - 1.0),
        "fit_seconds": float(time.perf_counter() - started),
        **{
            metric_name: (float(value) if pd.notna(value) else None)
            for metric_name, value in tm.items()
        },
    }

    with open(summary_path, "w", encoding="utf-8") as file:
        json.dump(summary, file, ensure_ascii=False, indent=2)
    weekly.to_csv(weekly_path, index=False)

    print(
        f"完成：{name} | {train_start}-{train_end} → {assess_start}-{assess_end} | "
        f"{summary['fit_seconds']:.1f}s"
    )
    del pred_frame
    gc.collect()
    return summary, weekly


## 13. Raw Core、Local FE 与 PCA 的统一比较

Stage 3 使用：

```text
Train 0–55
Assess 56–63
```

比较三类 package：

```text
A. Raw Core

B. Local FE
   unary_top10
   unary_top20
   unary + pair packages

C. PCA
   pca_numeric50
   pca_numeric100
```

另外保留一个可读的 `LOCAL_FE_REFERENCE_PACKAGE`，方便直接形成核心对照表：

```text
Raw Core
vs
Local FE Reference
vs
PCA-50 / PCA-100
```

每个 package 主要记录：

```text
Overall Gini
Mean Weekly Gini
Slope
Residual SD
Home Credit stability metric

Final design columns
PCA components
Explained variance ratio（描述信息）
Fit seconds（辅助参考）
```

这里要避免两个误解：

```text
更多 Feature Engineering
≠
一定更好

更高 explained variance
≠
一定更能预测 default
```

Stage 3 的作用只是把 package 数量进一步缩小；真正是否稳定，还要看后面的 F1–F4。


In [12]:
#@title 第 10 步：Stage 3 — Package Comparison
# ============================================================
# Stage-3 unified package evaluation
# ============================================================

PACKAGE_STAGE1_PATH = (
    NB11_CHECKPOINT_DIR
    / (
        "package_stage1_results_"
        f"{REPRESENTATION_EVALUATOR_VERSION}.csv"
    )
)

package_stage1 = pd.DataFrame()


if RUN_PACKAGE_STAGE1:

    rows = []

    for (
        name,
        package_def,
    ) in package_definitions.items():

        print(
            "Stage 3 方案：",
            name,
            "| 类型：",
            package_def[
                "mode"
            ],
        )

        summary, _ = evaluate_package(
            name,
            package_def,
            STAGE1_TRAIN_START,
            STAGE1_TRAIN_END,
            STAGE1_ASSESS_START,
            STAGE1_ASSESS_END,
        )

        rows.append(
            summary
        )


    package_stage1 = pd.DataFrame(
        rows
    )

    package_stage1.to_csv(
        PACKAGE_STAGE1_PATH,
        index=False,
    )


elif PACKAGE_STAGE1_PATH.exists():

    package_stage1 = pd.read_csv(
        PACKAGE_STAGE1_PATH
    )


if not package_stage1.empty:

    display(
        package_stage1
        .sort_values(
            [
                "stability_metric",
                "design_columns",
                "fit_seconds",
            ],
            ascending=[
                False,
                True,
                True,
            ],
        )
    )


    baseline_stage1 = (
        package_stage1
        .loc[
            package_stage1[
                "package"
            ].isin(
                BASELINE_PACKAGE_NAMES
            )
        ]
        .copy()
    )


    if not baseline_stage1.empty:

        print(
            "\n核心方案对比："
        )

        display(
            baseline_stage1
            .sort_values(
                [
                    "stability_metric",
                    "design_columns",
                ],
                ascending=[
                    False,
                    True,
                ],
            )[[
                "package",
                "package_mode",
                "gini",
                "mean_weekly_gini",
                "stability_metric",
                "design_columns",
                "representation_components",
                "explained_variance_ratio",
                "fit_seconds",
            ]]
        )


Stage 3 方案： raw_core | 类型： local_fe
checkpoint 命中：raw_core | 0-55 → 56-63
Stage 3 方案： unary_top10 | 类型： local_fe
checkpoint 命中：unary_top10 | 0-55 → 56-63
Stage 3 方案： unary_top20 | 类型： local_fe
checkpoint 命中：unary_top20 | 0-55 → 56-63
Stage 3 方案： unary_top30 | 类型： local_fe
checkpoint 命中：unary_top30 | 0-55 → 56-63
Stage 3 方案： unary_top40 | 类型： local_fe
checkpoint 命中：unary_top40 | 0-55 → 56-63
Stage 3 方案： unary_top10__pair_top10 | 类型： local_fe
checkpoint 命中：unary_top10__pair_top10 | 0-55 → 56-63
Stage 3 方案： unary_top10__pair_top20 | 类型： local_fe
[2026-10-07 17:20:25.801] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
完成：unary_top10__pair_top20 | 0-55 → 56-63 | 45.0s
Stage 3 方案： unary_top20__pair_top10 | 类型： local_fe
[2026-10-07 17:20:51.759] [CUML] [warning] L-BFGS stopped, because the line search failed to advance (step delta = 0.000000)
完成：unary_top20__pair_top10 | 0-55 → 56-63 | 24.1s
Stage 3 方案： unary_top20__pair_top20 | 类型： local_fe
[2026-10-07 1

,package,package_mode,evaluation_role,train_week_start,train_week_end,assess_week_start,assess_week_end,preprocessing_policy,preprocessing_fit_week_start,preprocessing_fit_week_end,...,design_columns,explained_variance_ratio,auc,gini,fit_seconds,mean_weekly_gini,weekly_gini_slope,negative_slope,weekly_residual_sd,stability_metric
5,unary_top10__pair_top10,local_fe,development_representation_search,0,55,56,63,nb7_full_development_frozen,0,63,...,394,NaN,0.798514,0.597027,17.243856,0.597394,0.000145,0.000000,0.016510,0.589139
6,unary_top10__pair_top20,local_fe,development_representation_search,0,55,56,63,nb7_full_development_frozen,0,63,...,411,NaN,0.803553,0.607106,45.003759,0.607334,-0.000344,0.000344,0.016708,0.568711
1,unary_top10,local_fe,development_representation_search,0,55,56,63,nb7_full_development_frozen,0,63,...,372,NaN,0.795733,0.591466,15.623526,0.591876,-0.000177,0.000177,0.018845,0.566913
17,pca_numeric100,pca_numeric,development_representation_search,0,55,56,63,nb7_full_development_frozen,0,63,...,318,1.000000,0.783533,0.567067,18.325516,0.567732,0.000575,0.000000,0.019340,0.558062
0,raw_core,local_fe,development_representation_search,0,55,56,63,nb7_full_development_frozen,0,63,...,318,NaN,0.783405,0.566809,21.450081,0.567458,0.000575,0.000000,0.019088,0.557914
16,pca_numeric85,pca_numeric,development_representation_search,0,55,56,63,nb7_full_development_frozen,0,63,...,306,0.992007,0.781431,0.562863,17.658922,0.563372,0.000014,0.000000,0.019562,0.553591
14,pca_numeric50,pca_numeric,development_representation_search,0,55,56,63,nb7_full_development_frozen,0,63,...,271,0.861625,0.779015,0.558030,13.320812,0.558661,0.000870,0.000000,0.018324,0.549499
13,pca_numeric25,pca_numeric,development_representation_search,0,55,56,63,nb7_full_development_frozen,0,63,...,246,0.637015,0.768365,0.536731,12.472203,0.537379,0.000098,0.000000,0.019333,0.527713
15,pca_numeric70,pca_numeric,development_representation_search,0,55,56,63,nb7_full_development_frozen,0,63,...,291,0.959978,0.781359,0.562719,14.032133,0.563167,-0.000377,0.000377,0.019328,0.520331
7,unary_top20__pair_top10,local_fe,development_representation_search,0,55,56,63,nb7_full_development_frozen,0,63,...,456,NaN,0.799992,0.599984,24.088629,0.600198,-0.001036,0.001036,0.014250,0.501868



核心方案对比：


,package,package_mode,gini,mean_weekly_gini,stability_metric,design_columns,representation_components,explained_variance_ratio,fit_seconds
17,pca_numeric100,pca_numeric,0.567067,0.567732,0.558062,318,97,1.000000,18.325516
0,raw_core,local_fe,0.566809,0.567458,0.557914,318,0,NaN,21.450081
14,pca_numeric50,pca_numeric,0.558030,0.558661,0.549499,271,50,0.861625,13.320812
7,unary_top20__pair_top10,local_fe,0.599984,0.600198,0.501868,456,0,NaN,24.088629


## 14. Stage 4 — F1–F4 Temporal Robustness

Stage 3 使用的是最近一个 development window：

```text
Train 0–55
Assess 56–63
```

F1–F4 则把少量 finalist packages 放回多个历史时间切分：

```text
F1  Train 0–31 → Assess 32–39
F2  Train 0–39 → Assess 40–47
F3  Train 0–47 → Assess 48–55
F4  Train 0–55 → Assess 56–63
```

其中 F4 与 Stage 3 使用同一个最近窗口，因此它不是一个新的独立验证集；F1–F3 主要提供 historical temporal robustness evidence。

这一阶段关注：

```text
mean fold Gini
worst fold Gini
fold-to-fold variation
mean stability metric
```

问题是：

> **最近窗口里表现好的 representation，在更早的时间窗口里是否也基本成立？**

### PCA 在每个 fold 都重新学习

PCA 的 reducer 不能先用 Week 0–63 学一次，再倒回 F1/F2 使用。

正确方式是：

```text
F1 → 只用 Week 0–31 学 PCA
F2 → 只用 Week 0–39 学 PCA
F3 → 只用 Week 0–47 学 PCA
F4 → 只用 Week 0–55 学 PCA
```

### NB7 preprocessing 与 PCA fit 是两个不同层次

NB11 延续 NB7 已经冻结的 preprocessing rule：

```text
median / mean / scale / category space
→ 来自 Development Week 0–63
→ 在 NB11 中固定复用
```

而 PCA reducer：

```text
→ 每个 temporal fold 只在当前 training window 内重新 fit
```

所以 F1–F4 的角色仍然是 **development temporal robustness diagnostic**，不是严格 nested preprocessing CV。

真正未参与 NB11 搜索的仍然是：

```text
Validation 64–77
OOT 78–91
```


In [13]:
#@title 第 11 步：F1–F4 Temporal Robustness
# ============================================================
# Select a small set of packages → F1–F4
# ============================================================

if not package_stage1.empty:

    # Keep the core comparison readable:
    # Raw Core / one Local-FE reference / PCA.
    forced_packages = [
        name
        for name in BASELINE_PACKAGE_NAMES
        if name in set(package_stage1["package"])
    ]

    remaining_ranked = (
        package_stage1
        .loc[
            ~package_stage1["package"].isin(forced_packages)
        ]
        .sort_values(
            ["stability_metric", "design_columns"],
            ascending=[False, True],
        )["package"]
        .tolist()
    )

    full_temporal_packages = list(
        dict.fromkeys([
            *forced_packages,
            *remaining_ranked,
        ])
    )[:MAX_FULL_TEMPORAL_PACKAGES]

else:

    full_temporal_packages = []


print(
    "进入 F1–F4 的方案：",
    full_temporal_packages,
)


FULL_TEMPORAL_PATH = (
    NB11_CHECKPOINT_DIR
    / (
        "package_full_temporal_results_"
        f"{REPRESENTATION_EVALUATOR_VERSION}.csv"
    )
)

package_full_temporal = pd.DataFrame()


if RUN_FULL_TEMPORAL:

    rows = []

    # Fold outermost: deterministic samples / DR blocks for one
    # temporal window stay hot while all promoted packages are checked.
    for (
        fold_name,
        tr0,
        tr1,
        va0,
        va1,
    ) in TEMPORAL_FOLDS:

        for package_name in full_temporal_packages:

            package_def = package_definitions[
                package_name
            ]

            print(
                package_name,
                fold_name,
            )

            summary, _ = evaluate_package(
                package_name,
                package_def,
                tr0,
                tr1,
                va0,
                va1,
            )

            rows.append({
                **summary,
                "fold": fold_name,
            })

    package_full_temporal = pd.DataFrame(
        rows
    )

    package_full_temporal.to_csv(
        FULL_TEMPORAL_PATH,
        index=False,
    )

elif FULL_TEMPORAL_PATH.exists():

    package_full_temporal = pd.read_csv(
        FULL_TEMPORAL_PATH
    )


if not package_full_temporal.empty:

    display(
        package_full_temporal.head(
            40
        )
    )


进入 F1–F4 的方案： ['raw_core', 'unary_top20__pair_top10', 'pca_numeric50', 'pca_numeric100', 'unary_top10__pair_top10', 'unary_top10__pair_top20', 'unary_top10', 'pca_numeric85']
raw_core F1
[2026-10-07 17:24:36.719] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
完成：raw_core | 0-31 → 32-39 | 5.3s
unary_top20__pair_top10 F1
[2026-10-07 17:24:49.102] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
完成：unary_top20__pair_top10 | 0-31 → 32-39 | 14.4s
pca_numeric50 F1
[2026-10-07 17:24:59.823] [CUML] [warning] L-BFGS stopped, because the line search failed to advance (step delta = 0.000000)
完成：pca_numeric50 | 0-31 → 32-39 | 9.0s
pca_numeric100 F1
[2026-10-07 17:25:09.370] [CUML] [warning] L-BFGS stopped, because the line search failed to advance (step delta = 0.000000)
完成：pca_numeric100 | 0-31 → 32-39 | 9.8s
unary_top10__pair_top10 F1
[2026-10-07 17:25:20.948] [CUML] [warning] L-BFGS stopped, because the line search failed

,package,package_mode,evaluation_role,train_week_start,train_week_end,assess_week_start,assess_week_end,preprocessing_policy,preprocessing_fit_week_start,preprocessing_fit_week_end,...,explained_variance_ratio,auc,gini,fit_seconds,mean_weekly_gini,weekly_gini_slope,negative_slope,weekly_residual_sd,stability_metric,fold
0,raw_core,local_fe,development_representation_search,0,31,32,39,nb7_full_development_frozen,0,63,...,NaN,0.703036,0.406072,5.282030,0.436066,-0.025617,0.025617,0.066625,-1.851543,F1
1,unary_top20__pair_top10,local_fe,development_representation_search,0,31,32,39,nb7_full_development_frozen,0,63,...,NaN,0.744391,0.488782,14.403935,0.505195,-0.019666,0.019666,0.044847,-1.247801,F1
2,pca_numeric50,pca_numeric,development_representation_search,0,31,32,39,nb7_full_development_frozen,0,63,...,0.912337,0.701143,0.402287,9.010371,0.435877,-0.025573,0.025573,0.066308,-1.847716,F1
3,pca_numeric100,pca_numeric,development_representation_search,0,31,32,39,nb7_full_development_frozen,0,63,...,1.000000,0.705334,0.410669,9.799146,0.443705,-0.025227,0.025227,0.066112,-1.809350,F1
4,unary_top10__pair_top10,local_fe,development_representation_search,0,31,32,39,nb7_full_development_frozen,0,63,...,NaN,0.729223,0.458446,11.699289,0.481837,-0.022123,0.022123,0.053187,-1.491537,F1
5,unary_top10__pair_top20,local_fe,development_representation_search,0,31,32,39,nb7_full_development_frozen,0,63,...,NaN,0.737303,0.474607,13.118215,0.497875,-0.022360,0.022360,0.050245,-1.494890,F1
6,unary_top10,local_fe,development_representation_search,0,31,32,39,nb7_full_development_frozen,0,63,...,NaN,0.729888,0.459775,7.727844,0.480254,-0.023012,0.023012,0.051739,-1.570700,F1
7,pca_numeric85,pca_numeric,development_representation_search,0,31,32,39,nb7_full_development_frozen,0,63,...,0.999793,0.706330,0.412660,10.744477,0.445147,-0.024890,0.024890,0.065910,-1.778137,F1
8,raw_core,local_fe,development_representation_search,0,39,40,47,nb7_full_development_frozen,0,63,...,NaN,0.788466,0.576931,6.594055,0.575907,0.002478,0.000000,0.022917,0.564449,F2
9,unary_top20__pair_top10,local_fe,development_representation_search,0,39,40,47,nb7_full_development_frozen,0,63,...,NaN,0.797673,0.595346,17.725827,0.593852,0.003405,0.000000,0.023773,0.581966,F2


## 15. Final Representation Comparison

F1–F4 后，每个 package 最重要的是：

```text
mean fold Gini
worst fold Gini
mean stability metric
fold-to-fold SD
final design columns
```

`fit seconds` 可以保留作参考，但不把它变成复杂的工程评分。

这里不再构造多目标 Pareto frontier。

最终直接看：

```text
1. 预测表现是否有增量
2. 时间稳定性是否更好或至少没有明显变差
3. 如果表现接近，哪一个 representation 更简单
```

因此可能出现：

```text
A. Raw Core 已经足够好
B. 少量 Local FE 有稳定增益
C. Pair / interaction 有额外价值
D. PCA 用更少维度保持了接近表现
```

Notebook 11 的目标不是证明某一种方法一定有效，而是回答：

> **哪一种 representation 值得进入 Notebook 12。**


In [14]:
#@title 第 12 步：汇总 F1–F4 结果
# ============================================================
# Final temporal package summary
# ============================================================

def summarize_full_temporal(table):

    if table.empty:
        return pd.DataFrame()

    out = (
        table
        .groupby(
            "package",
            as_index=False,
        )
        .agg(
            package_mode=(
                "package_mode",
                "first",
            ),
            evaluation_role=(
                "evaluation_role",
                "first",
            ),
            preprocessing_policy=(
                "preprocessing_policy",
                "first",
            ),
            reducer_fit_policy=(
                "reducer_fit_policy",
                "first",
            ),
            fold_count=(
                "fold",
                "nunique",
            ),
            mean_fold_gini=(
                "gini",
                "mean",
            ),
            worst_fold_gini=(
                "gini",
                "min",
            ),
            mean_stability_metric=(
                "stability_metric",
                "mean",
            ),
            worst_stability_metric=(
                "stability_metric",
                "min",
            ),
            fold_gini_sd=(
                "gini",
                "std",
            ),
            design_columns=(
                "design_columns",
                "max",
            ),
            representation_components=(
                "representation_components",
                "max",
            ),
            mean_explained_variance_ratio=(
                "explained_variance_ratio",
                "mean",
            ),
            mean_fit_seconds=(
                "fit_seconds",
                "mean",
            ),
        )
    )

    out["fold_gini_sd"] = (
        out["fold_gini_sd"]
        .fillna(0.0)
    )

    return out


final_temporal_summary = summarize_full_temporal(
    package_full_temporal
)


if not final_temporal_summary.empty:

    final_temporal_summary.to_csv(
        NB11_OUTPUT_DIR
        / "lr_representation_temporal_summary.csv",
        index=False,
    )

    display(
        final_temporal_summary
        .sort_values(
            [
                "mean_stability_metric",
                "worst_fold_gini",
                "design_columns",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
    )


,package,package_mode,evaluation_role,preprocessing_policy,reducer_fit_policy,fold_count,mean_fold_gini,worst_fold_gini,mean_stability_metric,worst_stability_metric,fold_gini_sd,design_columns,representation_components,mean_explained_variance_ratio,mean_fit_seconds
7,unary_top20__pair_top10,local_fe,development_representation_search,nb7_full_development_frozen,not_applicable,4,0.571777,0.488782,0.107179,-1.247801,0.055420,456,0,NaN,20.151882
6,unary_top10__pair_top20,local_fe,development_representation_search,nb7_full_development_frozen,not_applicable,4,0.563577,0.474607,-0.000724,-1.494890,0.061084,411,0,NaN,25.092180
5,unary_top10__pair_top10,local_fe,development_representation_search,nb7_full_development_frozen,not_applicable,4,0.551658,0.458446,-0.013591,-1.491537,0.064460,394,0,NaN,14.493596
4,unary_top10,local_fe,development_representation_search,nb7_full_development_frozen,not_applicable,4,0.550862,0.459775,-0.037390,-1.570700,0.061962,372,0,NaN,11.727634
0,pca_numeric100,pca_numeric,development_representation_search,nb7_full_development_frozen,current_training_window_only,4,0.535214,0.410669,-0.093380,-1.809350,0.083435,318,97,1.000000,14.270630
1,pca_numeric50,pca_numeric,development_representation_search,nb7_full_development_frozen,current_training_window_only,4,0.525776,0.402287,-0.102336,-1.847716,0.082620,271,50,0.881986,11.251698
2,pca_numeric85,pca_numeric,development_representation_search,nb7_full_development_frozen,current_training_window_only,4,0.532216,0.412660,-0.111313,-1.778137,0.080153,306,85,0.994560,14.334506
3,raw_core,local_fe,development_representation_search,nb7_full_development_frozen,not_applicable,4,0.493830,0.406072,-0.370266,-1.851543,0.090557,318,0,NaN,9.894369


In [15]:
#@title 第 13 步：Final Representation Comparison
# ============================================================
# Final representation comparison table
# ============================================================

final_representation_comparison = (
    final_temporal_summary
    .sort_values(
        [
            "mean_stability_metric",
            "worst_fold_gini",
            "design_columns",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
    if not final_temporal_summary.empty
    else pd.DataFrame()
)


if not final_representation_comparison.empty:

    final_representation_comparison.to_csv(
        NB11_OUTPUT_DIR
        / "lr_representation_final_comparison.csv",
        index=False,
    )

    display(
        final_representation_comparison
    )


,package,package_mode,evaluation_role,preprocessing_policy,reducer_fit_policy,fold_count,mean_fold_gini,worst_fold_gini,mean_stability_metric,worst_stability_metric,fold_gini_sd,design_columns,representation_components,mean_explained_variance_ratio,mean_fit_seconds
0,unary_top20__pair_top10,local_fe,development_representation_search,nb7_full_development_frozen,not_applicable,4,0.571777,0.488782,0.107179,-1.247801,0.055420,456,0,NaN,20.151882
1,unary_top10__pair_top20,local_fe,development_representation_search,nb7_full_development_frozen,not_applicable,4,0.563577,0.474607,-0.000724,-1.494890,0.061084,411,0,NaN,25.092180
2,unary_top10__pair_top10,local_fe,development_representation_search,nb7_full_development_frozen,not_applicable,4,0.551658,0.458446,-0.013591,-1.491537,0.064460,394,0,NaN,14.493596
3,unary_top10,local_fe,development_representation_search,nb7_full_development_frozen,not_applicable,4,0.550862,0.459775,-0.037390,-1.570700,0.061962,372,0,NaN,11.727634
4,pca_numeric100,pca_numeric,development_representation_search,nb7_full_development_frozen,current_training_window_only,4,0.535214,0.410669,-0.093380,-1.809350,0.083435,318,97,1.000000,14.270630
5,pca_numeric50,pca_numeric,development_representation_search,nb7_full_development_frozen,current_training_window_only,4,0.525776,0.402287,-0.102336,-1.847716,0.082620,271,50,0.881986,11.251698
6,pca_numeric85,pca_numeric,development_representation_search,nb7_full_development_frozen,current_training_window_only,4,0.532216,0.412660,-0.111313,-1.778137,0.080153,306,85,0.994560,14.334506
7,raw_core,local_fe,development_representation_search,nb7_full_development_frozen,not_applicable,4,0.493830,0.406072,-0.370266,-1.851543,0.090557,318,0,NaN,9.894369


## 16. Freeze Guardrail：NB11 冻结的是 Representation，不是最终 LR Model

当一个 package 被选择以后，本册冻结的是：

```text
representation mode
+
raw source dependencies
+
local engineered candidate specs（如果有）
+
PCA n_components（如果最终选择 PCA）
```

例如：

```text
local_fe
→ Core + selected transformations

pca_numeric
→ numeric Core 压成 K 个 components
→ missing indicators / categorical One-Hot 保留
```

需要区分两种状态：

```text
NB7 preprocessing state
→ Development Week 0–63 的 frozen rule

PCA state
→ 每次重新训练时，只在当时可用的 training rows 上重新 fit
```

因此 Notebook 11 冻结的是：

```text
representation rule
```

而不是某一次 fold 学出来的 PCA loadings。

本册仍然 **不冻结**：

```text
C
penalty
solver
max_iter / tol
```

这些属于 Notebook 12。

最终人工 Freeze 时主要参考：

```text
lr_representation_final_comparison.csv
```

如果多个方案表现非常接近，优先选择更简单、维度更低、时间表现更稳定的 representation。


In [16]:
#@title 第 14 步：人工 Freeze 给 Notebook 12
# ============================================================
# 15. Manual final freeze for Notebook 12
# ============================================================

def serialize_spec(
    spec,
):

    return {
        "candidate_id": (
            spec[
                "candidate_id"
            ]
        ),
        "family": (
            spec[
                "family"
            ]
        ),
        "sources": list(
            spec[
                "sources"
            ]
        ),
        "params": dict(
            spec[
                "params"
            ]
        ),
        "reason": spec.get(
            "reason",
            "",
        ),
    }


def serialize_representation_rule(
    package_name,
    package_def,
):

    mode = package_def[
        "mode"
    ]


    if mode == "local_fe":

        return {
            "package": package_name,
            "mode": mode,
            "source_scope": (
                "lr_core_plus_local_transform_dependencies"
            ),
            "requested_components": None,
            "fit_state_policy": (
                "Fit transform states only on the "
                "current training window."
            ),
            "preprocessing_policy": (
                PREPROCESSING_POLICY
            ),
            "preprocessing_fit_weeks": [
                PREPROCESSING_FIT_WEEK_START,
                PREPROCESSING_FIT_WEEK_END,
            ],
        }


    rule = {
        "package": package_name,
        "mode": mode,
        "source_scope": (
            package_def.get(
                "source_scope",
                "lr_core_features",
            )
        ),
        "requested_components": int(
            package_def[
                "n_components"
            ]
        ),
        "fit_state_policy": (
            "Refit PCA only on the current "
            "training window; never fit on future "
            "Validation/OOT rows."
        ),
        "preprocessing_policy": (
            PREPROCESSING_POLICY
        ),
        "preprocessing_fit_weeks": [
            PREPROCESSING_FIT_WEEK_START,
            PREPROCESSING_FIT_WEEK_END,
        ],
        "reducer_fit_policy": (
            REDUCER_FIT_POLICY
        ),
    }


    if mode == "pca_numeric":

        rule[
            "side_channels"
        ] = (
            "NB7 的 missing indicator 与类别 One-Hot 保持原样，不参与 PCA 压缩。"
        )



    return rule


if FREEZE_FINAL_LR_DESIGN:

    if (
        FINAL_LR_PACKAGE
        not in package_definitions
    ):

        raise KeyError(
            "未知的 FINAL_LR_PACKAGE="
            f"{FINAL_LR_PACKAGE}. "
            "可选方案="
            f"{list(package_definitions)}"
        )


    if (
        not final_temporal_summary.empty
        and FINAL_LR_PACKAGE
        not in set(
            final_temporal_summary[
                "package"
            ]
        )
    ):

        raise RuntimeError(
            "FINAL_LR_PACKAGE 没有完成 "
            "F1–F4 full temporal evaluation，"
            "不能冻结。"
        )


    final_package_def = (
        package_definitions[
            FINAL_LR_PACKAGE
        ]
    )

    final_mode = (
        final_package_def[
            "mode"
        ]
    )


    if final_mode == "local_fe":

        final_ids = (
            final_package_def.get(
                "candidate_ids",
                [],
            )
        )

        final_specs = [
            spec_lookup[
                candidate_id
            ]
            for candidate_id
            in final_ids
        ]

        required_sources = sorted({
            source
            for spec
            in final_specs
            for source
            in spec[
                "sources"
            ]
        })

        final_raw_sources = sorted(
            set(
                lr_core_features
            )
            | set(
                required_sources
            )
        )

    else:

        final_ids = []
        final_specs = []

        # Global DR challengers deliberately respect the
        # NB10 LR Core boundary.
        final_raw_sources = sorted(
            set(
                lr_core_features
            )
        )


    final_representation_rule = (
        serialize_representation_rule(
            FINAL_LR_PACKAGE,
            final_package_def,
        )
    )


    with open(
        NB11_OUTPUT_DIR
        / "selected_features_lr_raw_sources.json",
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            final_raw_sources,
            file,
            ensure_ascii=False,
            indent=2,
        )


    with open(
        NB11_OUTPUT_DIR
        / "selected_features_lr_engineered_specs.json",
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            [
                serialize_spec(
                    spec
                )
                for spec
                in final_specs
            ],
            file,
            ensure_ascii=False,
            indent=2,
        )


    with open(
        NB11_OUTPUT_DIR
        / "selected_features_lr_representation_spec.json",
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            final_representation_rule,
            file,
            ensure_ascii=False,
            indent=2,
        )


    with open(
        NB11_OUTPUT_DIR
        / "lr_nb12_handoff.json",
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            {
                "final_package": (
                    FINAL_LR_PACKAGE
                ),
                "representation_mode": (
                    final_mode
                ),
                "raw_source_count": len(
                    final_raw_sources
                ),
                "engineered_candidate_count": len(
                    final_specs
                ),
                "representation_spec_file": (
                    "selected_features_lr_"
                    "representation_spec.json"
                ),
                "representation_frozen": True,
                "representation_fit_state_frozen": False,
                "preprocessing_policy": (
                    PREPROCESSING_POLICY
                ),
                "preprocessing_fit_weeks": [
                    PREPROCESSING_FIT_WEEK_START,
                    PREPROCESSING_FIT_WEEK_END,
                ],
                "reducer_fit_policy": (
                    REDUCER_FIT_POLICY
                ),
                "nb7_preprocessing_signature": (
                    NB7_PREPROCESSING_SIGNATURE
                ),
                "nb10_source_signature": (
                    NB10_SOURCE_SIGNATURE
                ),
                "model_hyperparameters_frozen": False,
                "next_notebook": (
                    "NB12 Model / "
                    "Hyperparameter Search"
                ),
                "validation_read": False,
                "oot_read": False,
            },
            file,
            ensure_ascii=False,
            indent=2,
        )


    print(
        "已冻结 LR representation：",
        FINAL_LR_PACKAGE,
    )

    print(
        "Representation mode：",
        final_mode,
    )

    print(
        "Raw sources：",
        len(
            final_raw_sources
        ),
    )

    print(
        "Engineered candidates：",
        len(
            final_specs
        ),
    )


else:

    print(
        "Manual freeze disabled. "
        "Run comparison first, then "
        "set FINAL_LR_PACKAGE."
    )


已冻结 LR representation： unary_top20__pair_top10
Representation mode： local_fe
Raw sources： 115
Engineered candidates： 30


## 17. 结果摘要与下游接口

Representation search 以 NB10 选出的 source space 为输入，依次执行候选变换生成、cheap probe、package comparison 和 F1–F4 时间稳健性评估。

候选路线包括：

- **Raw Core**：原始核心特征作为基准；
- **Local Feature Engineering**：对筛选后的变量构造 log、winsor、bins、spline、ratio 与 interaction 等表示；
- **PCA Challenger**：评估 numeric Core 的低维表示是否能够保持预测表现。

实际保存的运行输出显示，本次冻结的 LR representation 为 `unary_top20__pair_top10`（`local_fe`），包含 **115 个 raw sources** 和 **30 个 engineered candidate specifications**。这一选择只对应 *representation*，后续模型超参数由 Notebook 12 继续搜索。

主要交付文件：

```text
selected_features_lr_raw_sources.json
selected_features_lr_engineered_specs.json
selected_features_lr_representation_spec.json
lr_nb12_handoff.json
```

**评估边界：** 以上结论来自 development 内部切分；Week 64–77 Validation 与 Week 78–91 OOT 均不参与本 Notebook 的 representation search。部分候选在保存日志中触发了优化器警告；后续模型选择需要继续检查收敛状态。


## 18. 方法参考与本册采用方式

### High-dimensional screening

- Fan, J. & Lv, J. (2008). *Sure Independence Screening for Ultrahigh Dimensional Feature Space*. JRSS-B. DOI: `10.1111/j.1467-9868.2008.00674.x`  
  本册借用的是 **cheap screening → moderate-scale careful search → revisit possible omissions** 的总体思想；并不声称这里实现了其理论 SIS/ISIS 条件。

### Tree-guided rules / interactions

- Friedman, J. H. & Popescu, B. E. (2008). *Predictive Learning via Rule Ensembles*. Annals of Applied Statistics. DOI: `10.1214/07-AOAS148`  
  本册借用：Tree structure 可以作为 nonlinearity / interaction proposal mechanism，而不是穷举所有 interactions。

### Hierarchical interaction search

- Bien, J., Taylor, J. & Tibshirani, R. (2013). *A Lasso for Hierarchical Interactions*. Annals of Statistics. DOI: `10.1214/13-AOS1096`  
  本册借用：hierarchy 可以作为 interaction-space 的结构约束；这里只作为 search-budget guardrail。

### Selection stability

- Meinshausen, N. & Bühlmann, P. (2010). *Stability Selection*. JRSS-B. DOI: `10.1111/j.1467-9868.2010.00740.x`  
  本册借用：不要只相信一次 selection；本项目有明确时间顺序，因此更关注 temporal repeatability。

### Representation tools

- scikit-learn `SplineTransformer`：建立受控的非线性 spline basis。
- scikit-learn `PCA`：对 numeric Core 做线性降维。
- scikit-learn `LogisticRegression`：作为 NB11 固定 learner；正则化与 solver 搜索留给 NB12。

`explained_variance_ratio` 只用于描述 PCA 压缩了多少输入变化，不把它当成 target predictive value；是否晋级仍由 Gini、temporal stability 与 representation dimension 决定。

### Evaluation alignment

Home Credit — Credit Risk Model Stability 的核心评价结构是：

```text
weekly Gini
→ mean
→ negative slope penalty
→ residual volatility penalty
```

NB11 延续这个结构来比较 representation 的时间稳定性。

### Selection bias guardrail

同一批数据如果被不断用于选择配置，性能估计会越来越乐观。因此 NB11 只在 Development Week 0–63 内完成 representation search，并继续封存：

```text
Validation 64–77
OOT 78–91
```

真正 future evaluation 的时间方向保持：

```text
Development 0–63
→ Validation 64–77
→ OOT 78–91
```
